# LunarLander multilevel LTLf — configurable multi-epsilon DDQN

Notebook Kaggle autosufficiente per confrontare due strategie multi-epsilon
(`cascade` con soglia oppure `visited` per stato utilizzato) e due architetture
DDQN (`classic` con un’unica uscita oppure `multi-head` con una testa per stato
DFA). Le scelte sono indipendenti, mentre replay buffer, reward e training loop
rimangono comuni alle quattro configurazioni.

Abilitare una GPU da **Settings → Accelerator → GPU** prima dell’esecuzione.


## 1. Install system and Python dependencies

In [ ]:

!apt-get update -qq
!apt-get install -y -qq mona graphviz swig
%pip install -q "gymnasium[box2d]" ltlf2dfa graphviz pandas matplotlib

## 2. Create the writable project directory

In [ ]:
from pathlib import Path
import os

WORK_DIR = Path("/kaggle/working/multilevel_multieps")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)
print(f"Working directory: {WORK_DIR}")


## 3. Write abstraction configuration and mapping utilities

In [ ]:
%%writefile abstraction.py
"""Configuration and mappings for a hierarchy of rectangular grid abstractions.

All mappings operate in the normalised square ``[0, 1] x [0, 1]``.  This makes
them independent of LunarLander's observation bounds and, importantly, allows
both the source and destination grids to have arbitrary dimensions.
"""

from __future__ import annotations

import json
import math
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class GridLevel:
    """One configured abstraction level."""

    width: int
    height: int
    name: str

    def __post_init__(self):
        if isinstance(self.width, bool) or not isinstance(self.width, int) or self.width <= 0:
            raise ValueError(f"{self.name}.grid_w must be a positive integer")
        if isinstance(self.height, bool) or not isinstance(self.height, int) or self.height <= 0:
            raise ValueError(f"{self.name}.grid_h must be a positive integer")
        if not isinstance(self.name, str) or not self.name.strip():
            raise ValueError("Every abstraction level must have a non-empty name")

    @property
    def shape(self):
        """Return ``(width, height)`` for mapping helpers."""
        return self.width, self.height


@dataclass(frozen=True)
class AbstractionConfig:
    """Validated ordered collection of abstraction levels.

    ``levels[0]`` is always the abstraction used by the automaton and training.
    The remaining levels are ordered dependencies: states at level *i* are
    mapped online to level *i + 1*, whose V-function is used as potential.
    """

    levels: tuple[GridLevel, ...]
    inter_level_shaping_scale: float = 1.0

    def __post_init__(self):
        if not self.levels:
            raise ValueError("abstraction.json must define at least one level")
        names = [level.name for level in self.levels]
        if len(names) != len(set(names)):
            raise ValueError("Abstraction level names must be unique")
        if (
            isinstance(self.inter_level_shaping_scale, bool)
            or not isinstance(self.inter_level_shaping_scale, (int, float))
            or not math.isfinite(self.inter_level_shaping_scale)
        ):
            raise ValueError("inter_level_shaping_scale must be a finite number")
        object.__setattr__(
            self,
            "inter_level_shaping_scale",
            float(self.inter_level_shaping_scale),
        )

    @property
    def primary(self):
        """Return level 1, whose coordinates define the automaton semantics."""
        return self.levels[0]

    @classmethod
    def from_dict(cls, data):
        if not isinstance(data, dict):
            raise ValueError("The abstraction configuration must be a JSON object")
        raw_levels = data.get("levels")
        if not isinstance(raw_levels, list) or not raw_levels:
            raise ValueError("abstraction.json must contain a non-empty 'levels' array")

        levels = []
        for index, raw_level in enumerate(raw_levels, start=1):
            if not isinstance(raw_level, dict):
                raise ValueError(f"levels[{index - 1}] must be a JSON object")
            name = raw_level.get("name", f"level{index}")
            width = raw_level.get("grid_w", raw_level.get("width"))
            height = raw_level.get("grid_h", raw_level.get("height"))
            if width is None or height is None:
                raise ValueError(
                    f"{name} must define grid_w/grid_h (or width/height)"
                )
            levels.append(GridLevel(width=width, height=height, name=name))
        return cls(
            tuple(levels),
            inter_level_shaping_scale=data.get(
                "inter_level_shaping_scale",
                1.0,
            ),
        )

    @classmethod
    def load(cls, path):
        """Load and validate an ``abstraction.json`` file."""
        path = Path(path)
        with path.open(encoding="utf-8") as config_file:
            return cls.from_dict(json.load(config_file))


def _validate_dimensions(width, height, label):
    if (
        isinstance(width, bool)
        or not isinstance(width, int)
        or isinstance(height, bool)
        or not isinstance(height, int)
        or width <= 0
        or height <= 0
    ):
        raise ValueError(f"{label} dimensions must be positive integers")


def _validate_cell(cell, width, height, label="source"):
    if not isinstance(cell, (tuple, list)) or len(cell) != 2:
        raise ValueError("A grid cell must contain exactly two coordinates")
    x, y = cell
    if isinstance(x, bool) or not isinstance(x, int):
        raise ValueError("Grid x-coordinate must be an integer")
    if isinstance(y, bool) or not isinstance(y, int):
        raise ValueError("Grid y-coordinate must be an integer")
    if not 0 <= x < width or not 0 <= y < height:
        raise ValueError(
            f"Cell ({x}, {y}) is outside the {label} grid {width}x{height}"
        )
    return x, y


def map_cell(
    cell,
    source_width,
    source_height,
    target_width,
    target_height,
):
    """Map one cell centre between arbitrary rectangular grids.

    The same function is used in either direction by swapping source and target
    dimensions.  A centre-based mapping is deterministic even when a coarse
    cell overlaps multiple finer cells.
    """
    _validate_dimensions(source_width, source_height, "Source")
    _validate_dimensions(target_width, target_height, "Target")
    x, y = _validate_cell(cell, source_width, source_height)
    target_x = min(
        int(((x + 0.5) / source_width) * target_width),
        target_width - 1,
    )
    target_y = min(
        int(((y + 0.5) / source_height) * target_height),
        target_height - 1,
    )
    return target_x, target_y


def map_state(
    state,
    source_width,
    source_height,
    target_width,
    target_height,
):
    """Map the spatial part of ``(x, y, q)`` while preserving DFA state ``q``."""
    if not isinstance(state, (tuple, list)) or len(state) != 3:
        raise ValueError("An abstract state must be (x, y, q)")
    x, y = map_cell(
        state[:2],
        source_width,
        source_height,
        target_width,
        target_height,
    )
    return x, y, state[2]


def overlapping_cells(
    cell,
    source_width,
    source_height,
    target_width,
    target_height,
):
    """Return every destination cell with positive area overlap.

    This is the set-valued counterpart of :func:`map_cell`, useful for exact
    coarse-to-fine and fine-to-coarse relationships.
    """
    _validate_dimensions(source_width, source_height, "Source")
    _validate_dimensions(target_width, target_height, "Target")
    x, y = _validate_cell(cell, source_width, source_height)

    min_x = int(math.floor(x * target_width / source_width))
    max_x = int(math.ceil((x + 1) * target_width / source_width) - 1)
    min_y = int(math.floor(y * target_height / source_height))
    max_y = int(math.ceil((y + 1) * target_height / source_height) - 1)
    return [
        (target_x, target_y)
        for target_x in range(max(0, min_x), min(target_width - 1, max_x) + 1)
        for target_y in range(max(0, min_y), min(target_height - 1, max_y) + 1)
    ]


def map_waypoints(
    waypoints,
    source_width,
    source_height,
    target_width,
    target_height,
):
    """Map a proposition-to-cell dictionary between arbitrary grids."""
    return {
        name: map_cell(
            tuple(coordinates),
            source_width,
            source_height,
            target_width,
            target_height,
        )
        for name, coordinates in waypoints.items()
    }


## 4. Write the multilevel abstract MDP and LTLf automaton

In [ ]:
%%writefile abstract_mdps.py
import re
import warnings
from collections import defaultdict

from abstraction import map_state, map_waypoints

class LTLfAutomaton:
    """
    Wrap ltlf2dfa and expose its DFA as a graph that can be traversed by the MDP.
    """
    def __init__(self, formula_str):
        # Keep MDP/mapping utilities importable without the optional DFA parser.
        from ltlf2dfa.parser.ltlf import LTLfParser

        self.formula_str = formula_str
        
        # Parse the formula and generate its DFA in DOT format.
        parser = LTLfParser()
        parsed_formula = parser(formula_str)
        dot_string = parsed_formula.to_dfa()
        self.dot_string = parsed_formula.to_dfa()
        
        # Initialize the automaton data structures.
        self.states = set()
        self.accepting_states = set()
        self.transitions = {}  # {source_state: [(Boolean_guard, destination_state), ...]}
        self.initial_state = None
        
        # Extract states and transitions from the DOT representation.
        self._parse_dot(dot_string)
        
        # Keep a stable state order for the MDP and one-hot encodings.
        self.states = sorted(list(self.states))
        self.num_phases = len(self.states)

    def _parse_dot(self, dot_string):
        """
        Parse the DOT output and extract states, accepting states, the initial
        state, and guarded transitions.
        """
        # Extract accepting states, e.g. node [shape = doublecircle]; 2 3;.
        match_acc = re.search(r'node\s*\[shape\s*=\s*doublecircle\]\s*;\s*(.*?);', dot_string)
        if match_acc:
            acc_str = match_acc.group(1).replace(',', ' ')
            self.accepting_states = set(int(s) for s in acc_str.split() if s.strip().isdigit())
            
        # Extract guarded transitions, e.g. 1 -> 2 [label="wp1 & ~wp2"].
        trans_matches = re.findall(r'(\d+)\s*->\s*(\d+)\s*\[label\s*=\s*"(.*?)"\]', dot_string)
        for src_str, dst_str, guard in trans_matches:
            src = int(src_str)
            dst = int(dst_str)
            self.states.add(src)
            self.states.add(dst)
            
            if src not in self.transitions:
                self.transitions[src] = []
            self.transitions[src].append((guard, dst))
            
        # Extract the initial state from the unlabeled edge leaving the invisible node.
        # Example: 0 [style=invis]; 0 -> 1;.
        init_match = re.search(r'(\d+)\s*->\s*(\d+)\s*;', dot_string)
        if init_match:
            self.initial_state = int(init_match.group(2))
        else:
            self.initial_state = min(self.states) if self.states else 0

    def get_initial_q(self):
        """Return the identifier of the DFA pre-trace state."""
        return self.initial_state

    def is_goal_reached(self, current_q):
        """Return whether the current DFA state is accepting."""
        return current_q in self.accepting_states

    def get_next_q(self, current_q, truth_assignment):
        """
        Evaluate outgoing transition guards and return the next DFA state.
        """
        if current_q not in self.transitions:
            return current_q
            
        for guard, next_q in self.transitions[current_q]:
            if self._eval_guard(guard, truth_assignment):
                return next_q
                
        return current_q

    def _eval_guard(self, guard, truth_assignment):
        """
        Convert a DOT guard such as "wp1 & ~wp2" to Python syntax and evaluate
        it against the current truth assignment.
        """
        guard = guard.strip()
        
        # Handle numeric and textual Boolean constants.
        if guard.lower() in ["1", "true"]: return True
        if guard.lower() in ["0", "false"]: return False
        
        # Convert the standard Boolean operators to Python syntax.
        expr = guard.replace('&', ' and ').replace('|', ' or ').replace('~', ' not ').replace('!', ' not ')
        
        try:
            # Disable built-ins while evaluating the Boolean expression.
            return eval(expr, {"__builtins__": {}}, truth_assignment)
        except Exception as e:
            print(f"[LTLfAutomaton error] Could not evaluate transition guard '{guard}': {e}")
            return False

    def render_graph(self, filename="ltlf_automaton", directory="img"):
        """Render the DFA and save it as a PNG image."""
        try:
            from graphviz import Source

            # ltlf2dfa emits a left-to-right graph.  With complex formulae the
            # transition guards become wide, leaving the resulting PNG only a
            # few pixels high.  A top-to-bottom layout gives labels enough room
            # and keeps the automaton readable independently of formula length.
            render_dot = re.sub(
                r"rankdir\s*=\s*LR\s*;",
                "rankdir = TB;",
                self.dot_string,
                count=1,
            )
            render_dot = re.sub(
                r"(digraph[^{]*\{)",
                (
                    r"\1\n"
                    r'graph [pad="0.35", nodesep="0.55", ranksep="0.75"];' "\n"
                    r'node [width="0.55", height="0.55"];' "\n"
                    r'edge [fontsize="10"];'
                ),
                render_dot,
                count=1,
            )
            src = Source(render_dot)
            src.render(filename=filename, directory=directory, format='png', cleanup=True)
            print(f"Automaton graph saved to: {directory}/{filename}.png")
        except Exception as e:
            print(f"[Graphviz error] Could not render the automaton graph: {e}")


class LTLfWaypointMDP:
    """
    Abstract MDP guided by an LTLf automaton.
    Each abstract state is (x, y, q), where q is the DFA state identifier.
    """
    def __init__(
        self,
        waypoints_dict,
        ltlf_automaton,
        width=12,
        height=12,
        gamma=0.99,
        goal_reward=10000,
        level_name="level1",
    ):
        if width <= 0 or height <= 0:
            raise ValueError("Abstract grid dimensions must be positive")
        self.width = width
        self.height = height
        self.level_name = level_name
        self.gamma = gamma
        self.actions = [0, 1, 2, 3, 4, 5, 6, 7] # Include diagonal movements.
        
        self.waypoints_dict = waypoints_dict
        self.automaton = ltlf_automaton
        self.num_phases = self.automaton.num_phases
        
        # Generate every combination of grid position and DFA state.
        self.states = [(x, y, q) for x in range(width) for y in range(height) for q in self.automaton.states]
        
        self.goal_reward = goal_reward
        self.v_star = defaultdict(float)
        self.upper_level_mdp = None
        self.inter_level_shaping_scale = 0.0
        self.value_iteration_iterations = 0
        
    def _get_truth_assignment(self, x, y):
        """
        Map the current grid coordinates to a Boolean proposition assignment.
        """
        truth_assignment = {}
        for prop_name, (wp_x, wp_y) in self.waypoints_dict.items():
            truth_assignment[prop_name] = (x == wp_x and y == wp_y)
        return truth_assignment

    def get_transitions(self, state, action):
        x, y, q = state
        
        # Apply the abstract physical movement.
        next_y = y
        if action in [0, 4, 5]:    next_y = min(y + 1, self.height - 1)
        elif action in [1, 6, 7]:  next_y = max(y - 1, 0)
            
        next_x = x
        if action in [2, 4, 6]:    next_x = max(x - 1, 0)
        elif action in [3, 5, 7]:  next_x = min(x + 1, self.width - 1)
        
        # Evaluate propositions at the arrival coordinates.
        truth_assignment = self._get_truth_assignment(next_x, next_y)
        
        # Advance the automaton using the arrival-state valuation.
        next_q = self.automaton.get_next_q(q, truth_assignment)

        next_state = (next_x, next_y, next_q)
        # Match the single-level LunarLander convention: abstract transitions
        # never emit reward. The task reward is represented by the boundary
        # value assigned to accepting product states during value iteration.
        return next_state, 0.0

    def map_state_to_upper_level(self, state):
        """Map a state and apply the upper level's coarser proposition labels.

        A waypoint labels its complete macro-cell in the upper abstraction.
        Consequently the mapped DFA state must consume that macro-cell's
        valuation instead of blindly preserving the lower-level ``q``.
        """
        if self.upper_level_mdp is None:
            raise ValueError(f"{self.level_name} has no upper abstraction level")
        upper_state = map_state(
            state,
            source_width=self.width,
            source_height=self.height,
            target_width=self.upper_level_mdp.width,
            target_height=self.upper_level_mdp.height,
        )
        upper_x, upper_y, q = upper_state
        upper_truth_assignment = (
            self.upper_level_mdp._get_truth_assignment(upper_x, upper_y)
        )
        upper_q = self.automaton.get_next_q(q, upper_truth_assignment)
        return upper_x, upper_y, upper_q

    def get_upper_level_potential(self, state):
        """Map ``state`` canonically and read the upper-level V*."""
        if self.upper_level_mdp is None:
            return 0.0
        upper_state = self.map_state_to_upper_level(state)
        return self.upper_level_mdp.v_star.get(upper_state, 0.0)

    def get_inter_level_shaping_reward(self, state, next_state):
        """Return K * (gamma * V_upper(map(s')) - V_upper(map(s)))."""
        if self.upper_level_mdp is None:
            return 0.0
        state_potential = self.get_upper_level_potential(state)
        next_state_potential = self.get_upper_level_potential(next_state)
        return self.inter_level_shaping_scale * (
            self.gamma * next_state_potential
            - state_potential
        )

    def print_policy(self):
        arrows = {
            0: "↑",
            1: "↓",
            2: "←",
            3: "→",
            4: "↖",
            5: "↗",
            6: "↙",
            7: "↘"
        }

        for q in self.automaton.states:
            print(f"\n===== POLICY - DFA STATE q={q} =====")

            for y in reversed(range(self.height)):
                row = []

                for x in range(self.width):
                    state = (x, y, q)

                    if self.automaton.is_goal_reached(q):
                        row.append(" G ")
                        continue

                    best_action = None
                    best_value = -float("inf")

                    for a in self.actions:
                        next_state, reward = self.get_transitions(state, a)
                        shaping_reward = self.get_inter_level_shaping_reward(
                            state,
                            next_state,
                        )
                        value = (
                            reward
                            + shaping_reward
                            + self.gamma * self.v_star[next_state]
                        )

                        if value > best_value:
                            best_value = value
                            best_action = a

                    row.append(f" {arrows[best_action]} ")

                print("".join(row))
    
    def value_iteration(
        self,
        theta=0.001,
        upper_level_mdp=None,
        shaping_scale=1.0,
        print_policy=True,
    ):
        """Compute V*, reading an optional upper-level potential online."""
        if theta <= 0:
            raise ValueError("theta must be greater than zero")

        self.upper_level_mdp = upper_level_mdp
        self.inter_level_shaping_scale = (
            float(shaping_scale) if self.upper_level_mdp is not None else 0.0
        )
        # Upper-level values are neither copied nor used as an initialisation:
        # each PBRS evaluation maps the current states and reads upper V* online.
        self.v_star = defaultdict(float)
        shaping_label = (
            f", online PBRS K={self.inter_level_shaping_scale:g}"
            if self.upper_level_mdp is not None
            else ", no inter-level shaping"
        )
        print(
            f"Value Iteration [{self.level_name}: "
            f"{self.width}x{self.height}{shaping_label}]..."
        )

        for s in self.states:
            if self.automaton.is_goal_reached(s[2]):
                self.v_star[s] = self.goal_reward

        iterations = 0
        while True:
            iterations += 1
            delta = 0
            new_v = self.v_star.copy()
            for s in self.states:
                if not self.automaton.is_goal_reached(s[2]):
                    v_actions = []
                    for action in self.actions:
                        next_state, reward = self.get_transitions(s, action)
                        shaping_reward = self.get_inter_level_shaping_reward(
                            s,
                            next_state,
                        )
                        v_actions.append(
                            reward
                            + shaping_reward
                            + self.gamma * self.v_star[next_state]
                        )
                    best_v = max(v_actions)
                    delta = max(delta, abs(best_v - self.v_star[s]))
                    new_v[s] = best_v
            self.v_star = new_v
            if delta < theta:
                break

        self.value_iteration_iterations = iterations
        if print_policy:
            self.print_policy()
        return self.v_star


class MultiLevelWaypointMDP:
    """Ordered hierarchy of grid MDPs sharing one LTLf automaton.

    Waypoints and automaton interaction are defined on level 1.  Waypoint
    coordinates for every other grid are projections of those coordinates.
    Value iteration runs from the final level back to level 1; each level is
    shaped through online mapped lookups into V* of the following level.
    """

    def __init__(
        self,
        waypoints_dict,
        ltlf_automaton,
        abstraction_config,
        gamma=0.99,
        goal_reward=10000,
    ):
        self.automaton = ltlf_automaton
        self.abstraction_config = abstraction_config
        self.gamma = gamma
        self.goal_reward = goal_reward
        self.levels = []

        primary = abstraction_config.primary
        primary_waypoints = {
            name: tuple(coordinates) for name, coordinates in waypoints_dict.items()
        }
        for index, level in enumerate(abstraction_config.levels):
            if index == 0:
                level_waypoints = primary_waypoints
            else:
                level_waypoints = map_waypoints(
                    primary_waypoints,
                    primary.width,
                    primary.height,
                    level.width,
                    level.height,
                )
                self._warn_on_waypoint_collisions(level.name, level_waypoints)
            self.levels.append(
                LTLfWaypointMDP(
                    waypoints_dict=level_waypoints,
                    ltlf_automaton=ltlf_automaton,
                    width=level.width,
                    height=level.height,
                    gamma=gamma,
                    goal_reward=goal_reward,
                    level_name=level.name,
                )
            )

    @staticmethod
    def _warn_on_waypoint_collisions(level_name, waypoints):
        cells_to_names = defaultdict(list)
        for name, cell in waypoints.items():
            cells_to_names[cell].append(name)
        collisions = {
            cell: names for cell, names in cells_to_names.items() if len(names) > 1
        }
        if collisions:
            warnings.warn(
                f"{level_name} maps multiple propositions to the same cells: "
                f"{collisions}. They will be true simultaneously on that level.",
                UserWarning,
                stacklevel=3,
            )

    @property
    def primary_mdp(self):
        """Return level 1, used unchanged by automaton handling and training."""
        return self.levels[0]

    def compute_value_functions(self, theta=0.001, print_policies=False):
        """Compute every V-function with recursive inter-level PBRS."""
        following_mdp = None
        for current_mdp in reversed(self.levels):
            current_mdp.value_iteration(
                theta=theta,
                upper_level_mdp=following_mdp,
                shaping_scale=(
                    self.abstraction_config.inter_level_shaping_scale
                ),
                print_policy=print_policies,
            )
            following_mdp = current_mdp
        return [level.v_star for level in self.levels]


## 5. Write the configurable classic/multi-head DDQN agent


In [ ]:
%%writefile agent.py
import numpy as np
import random as ran
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F


class QNetwork(nn.Module):
    """Classic shared Q-network receiving the physical state and DFA one-hot."""

    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, action_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


class DuelingQNetwork(nn.Module):
    """Classic dueling Q-network receiving the augmented state."""

    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.feature = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
        )
        self.value_stream = nn.Linear(128, 1)
        self.advantage_stream = nn.Linear(128, action_dim)

    def forward(self, x):
        features = self.feature(x)
        value = self.value_stream(features)
        advantages = self.advantage_stream(features)
        return value + advantages - advantages.mean(dim=1, keepdim=True)


def _select_head_outputs(heads, features, head_indices):
    """Evaluate all heads and select one output row per batch element."""
    if head_indices.ndim != 1:
        raise ValueError("head_indices must be a one-dimensional tensor")
    if features.shape[0] != head_indices.shape[0]:
        raise ValueError("one head index is required for each batch element")
    if torch.any(head_indices < 0) or torch.any(head_indices >= len(heads)):
        raise IndexError("multi-head index is out of range")

    all_outputs = torch.stack(
        [head(features) for head in heads],
        dim=1,
    )
    batch_indices = torch.arange(features.shape[0], device=features.device)
    return all_outputs[batch_indices, head_indices]


class MultiHeadQNetwork(nn.Module):
    """Shared physical-state encoder with one Q-value head per DFA state."""

    def __init__(self, state_dim, action_dim, num_heads):
        super().__init__()
        if num_heads <= 0:
            raise ValueError("num_heads must be greater than zero")
        self.feature = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
        )
        self.heads = nn.ModuleList(
            nn.Linear(128, action_dim) for _ in range(num_heads)
        )

    def forward(self, x, head_indices):
        features = self.feature(x)
        return _select_head_outputs(self.heads, features, head_indices)


class MultiHeadDuelingQNetwork(nn.Module):
    """Dueling DDQN variant with separate value/advantage heads per DFA state."""

    def __init__(self, state_dim, action_dim, num_heads):
        super().__init__()
        if num_heads <= 0:
            raise ValueError("num_heads must be greater than zero")
        self.feature = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
        )
        self.value_heads = nn.ModuleList(
            nn.Linear(128, 1) for _ in range(num_heads)
        )
        self.advantage_heads = nn.ModuleList(
            nn.Linear(128, action_dim) for _ in range(num_heads)
        )

    def forward(self, x, head_indices):
        features = self.feature(x)
        value = _select_head_outputs(
            self.value_heads,
            features,
            head_indices,
        )
        advantages = _select_head_outputs(
            self.advantage_heads,
            features,
            head_indices,
        )
        return value + advantages - advantages.mean(dim=1, keepdim=True)

class ReplayBuffer:
    def __init__(self, capacity, num_phases):
        if num_phases < 0:
            raise ValueError("num_phases cannot be negative")
        self.capacity = capacity
        self.num_phases = num_phases
        self.buffer = []
        self.phase_indices = []
        self.phase_counts = np.zeros(num_phases, dtype=np.int64)
        self.position = 0

    def push(self, state, action, reward, next_state, done):
        """Insert a transition and update DFA-state counts in constant time."""
        transition = (state, action, reward, next_state, done)
        phase_index = (
            int(np.argmax(state[-self.num_phases:]))
            if self.num_phases > 0
            else None
        )

        if len(self.buffer) < self.capacity:
            self.buffer.append(transition)
            self.phase_indices.append(phase_index)
        else:
            replaced_phase_index = self.phase_indices[self.position]
            if replaced_phase_index is not None:
                self.phase_counts[replaced_phase_index] -= 1
            self.buffer[self.position] = transition
            self.phase_indices[self.position] = phase_index

        if phase_index is not None:
            self.phase_counts[phase_index] += 1
        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        """Sample transitions efficiently from the indexable ring buffer."""
        batch = ran.sample(self.buffer, batch_size)
        state, action, reward, next_state, done = map(np.array, zip(*batch))
        return state, action, reward, next_state, done

    def __len__(self):
        return len(self.buffer)

    def q_fraction_onehot(self, q_index, num_phases):
        """Return a DFA-state fraction using incrementally maintained counts."""
        if num_phases != self.num_phases:
            raise ValueError(f"Expected {self.num_phases} DFA states, received {num_phases}")
        if not 0 <= q_index < self.num_phases:
            raise IndexError(f"DFA state index {q_index} is out of range")
        if len(self.buffer) == 0:
            return 0.0
        return float(self.phase_counts[q_index] / len(self.buffer))

class HierarchicalDQNLearner:
    def __init__(
        self,
        env,
        abstract_mdp=None,
        max_episodes=1000,
        eps_decay=0.995,
        gamma=0.99,
        policy_name="policy",
        extra_state_dims=0,
        use_polyak=True,
        tau=0.005,
        target_update_freq=1000,
        network_architecture="multi-head",
        network_type="standard",
    ):
        if extra_state_dims <= 0:
            raise ValueError("DFA-guided DDQN requires at least one DFA state")
        if not 0.0 < tau <= 1.0:
            raise ValueError("tau must be in the interval (0, 1]")
        if target_update_freq <= 0:
            raise ValueError("target_update_freq must be greater than zero")
        if network_architecture not in {"classic", "multi-head"}:
            raise ValueError("network_architecture must be one of: classic, multi-head")
        if network_type not in {"standard", "dueling"}:
            raise ValueError("network_type must be one of: standard, dueling")

        self.env = env
        self.abstract_mdp = abstract_mdp
        self.max_episodes = max_episodes
        self.gamma = gamma
        self.policy_name = policy_name
        self.network_architecture = network_architecture
        self.network_type = network_type
        base_name = "Dueling DDQN" if network_type == "dueling" else "DDQN"
        self.algo_name = (
            f"Multi-head {base_name}"
            if network_architecture == "multi-head"
            else f"Classic {base_name}"
        )
        self.num_heads = extra_state_dims
        self.observation_dim = self.env.observation_space.shape[0]
        
        self.batch_size = 64
        self.lr = 1e-3
        self.use_polyak = use_polyak
        self.tau = tau
        self.target_update_freq = target_update_freq
        self.optimization_steps = 0
        self.eps = 1.0
        self.eps_min = 0.01
        self.eps_decay = eps_decay
        
        # The classic network receives the augmented state. In the multi-head
        # architecture the trunk sees only the physical observation and the
        # one-hot is used exclusively to route samples to their heads.
        state_dim = (
            self.observation_dim + self.num_heads
            if network_architecture == "classic"
            else self.observation_dim
        )
        action_dim = self.env.action_space.n
        
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        if network_architecture == "multi-head":
            network_cls = MultiHeadDuelingQNetwork if network_type == "dueling" else MultiHeadQNetwork
            network_args = (state_dim, action_dim, self.num_heads)
        else:
            network_cls = DuelingQNetwork if network_type == "dueling" else QNetwork
            network_args = (state_dim, action_dim)
        self.policy_net = network_cls(*network_args).to(self.device)
        architecture_details = (
            f"{self.num_heads} heads"
            if network_architecture == "multi-head"
            else "one shared output"
        )
        print(f"Using device:{self.device} | Architecture: {self.algo_name} ({architecture_details})")
        
        self.target_net = network_cls(*network_args).to(self.device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()
        
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=self.lr)
        self.memory = ReplayBuffer(capacity=300000, num_phases=extra_state_dims)

    def _split_augmented_states(self, augmented_states):
        """Separate physical observations from their DFA head indices."""
        expected_width = self.observation_dim + self.num_heads
        if augmented_states.ndim != 2 or augmented_states.shape[1] != expected_width:
            raise ValueError(
                f"Expected augmented states with shape (batch, {expected_width})"
            )
        physical_states = augmented_states[:, :self.observation_dim]
        phase_one_hot = augmented_states[:, self.observation_dim:]
        head_indices = phase_one_hot.argmax(dim=1).long()
        return physical_states, head_indices

    def _network_values(self, network, augmented_states):
        """Evaluate either architecture through one common learner interface."""
        if self.network_architecture == "classic":
            return network(augmented_states)
        physical_states, head_indices = self._split_augmented_states(
            augmented_states
        )
        return network(physical_states, head_indices)

    def select_action(self, state):
        if ran.random() < self.eps:
            return self.env.action_space.sample()
        else:
            with torch.no_grad():
                augmented_state = torch.as_tensor(
                    state,
                    dtype=torch.float32,
                    device=self.device,
                ).unsqueeze(0)
                q_values = self._network_values(
                    self.policy_net,
                    augmented_state,
                )
                return q_values.argmax(dim=1).item()

    def optimize_model(self):
        if len(self.memory) < self.batch_size: return
            
        states, actions, rewards, next_states, dones = self.memory.sample(self.batch_size)
        
        augmented_states = torch.FloatTensor(states).to(self.device)
        actions = torch.LongTensor(actions).unsqueeze(1).to(self.device)
        rewards = torch.FloatTensor(rewards).unsqueeze(1).to(self.device)
        augmented_next_states = torch.FloatTensor(next_states).to(self.device)
        dones = torch.FloatTensor(dones).unsqueeze(1).to(self.device)

        q_values = self._network_values(
            self.policy_net,
            augmented_states,
        ).gather(1, actions)
        
        with torch.no_grad():
            best_actions = self._network_values(
                self.policy_net,
                augmented_next_states,
            ).argmax(dim=1).unsqueeze(1)
            next_q_values = self._network_values(
                self.target_net,
                augmented_next_states,
            ).gather(1, best_actions)
            target_q_values = rewards + (1 - dones) * self.gamma * next_q_values
            
        loss = F.mse_loss(q_values, target_q_values)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        self.optimization_steps += 1
        if self.use_polyak:
            for target_param, policy_param in zip(self.target_net.parameters(), self.policy_net.parameters()):
                target_param.data.copy_(self.tau * policy_param.data + (1.0 - self.tau) * target_param.data)
        elif self.optimization_steps % self.target_update_freq == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())

    def _save_policy(self):
        os.makedirs("./policy", exist_ok=True)
        torch.save(self.policy_net.state_dict(), f"./policy/{self.policy_name}")


## 6. Write plotting utilities

In [ ]:
%%writefile utils.py
"""Spatial mapping and plotting utilities for multi-epsilon training."""

# ==============================
# Standard library imports
# ==============================

import os

# ==============================
# External imports
# ==============================

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle


LEARNING_REWARD_COLOR = "#0072B2"
TASK_REWARD_COLOR = "#D55E00"
EPSILON_COLOR = "#E6AB02"
RAW_DATA_COLOR = "#777777"
REFERENCE_LINE_COLOR = "#666666"
SERIES_COLORS = (
    LEARNING_REWARD_COLOR,
    TASK_REWARD_COLOR,
    "#CC79A7",
    "#009E73",
    "#56B4E9",
    "#000000",
)
EPSILON_LINESTYLES = ("--", (0, (5, 2, 1, 2)), ":", "-.")
PAPER_COLORS = SERIES_COLORS

plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Liberation Sans", "DejaVu Sans"],
        "font.size": 10,
        "axes.labelsize": 10,
        "axes.linewidth": 0.8,
        "legend.fontsize": 9,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "xtick.direction": "in",
        "ytick.direction": "in",
        "savefig.dpi": 300,
        "savefig.facecolor": "white",
    }
)


# ==============================
# Spatial discretization and grid geometry
# ==============================

# Legacy discretization. To reactivate it, uncomment this function and comment
# out the active phi_mapping_grid implementation immediately below.
#
# def phi_mapping_grid(obs, grid_w=12, grid_h=12):
#     """Map coordinates using the original grid_size - 1 discretization."""
#     x, y = float(obs[0]), float(obs[1])
#     abstract_x = int(np.clip((x + 1.0) / 2.0 * (grid_w - 1), 0, grid_w - 1))
#     abstract_y = int(np.clip(y / 1.5 * (grid_h - 1), 0, grid_h - 1))
#     return abstract_x, abstract_y


def phi_mapping_grid(obs, grid_w=12, grid_h=12):
    """Map LunarLander coordinates to uniform bins over x=[-1,1], y=[0,1.5]."""
    if grid_w <= 0 or grid_h <= 0:
        raise ValueError("grid_w and grid_h must be positive")

    x, y = float(obs[0]), float(obs[1])
    abstract_x = int(np.floor((x + 1.0) / 2.0 * grid_w))
    abstract_y = int(np.floor(y / 1.5 * grid_h))
    abstract_x = int(np.clip(abstract_x, 0, grid_w - 1))
    abstract_y = int(np.clip(abstract_y, 0, grid_h - 1))
    return abstract_x, abstract_y


def _axis_boundaries(map_axis, size, lower, upper):
    """Infer bin boundaries from the active mapper."""
    if size <= 0:
        raise ValueError("grid dimensions must be positive")

    boundaries = [float(lower)]
    iterations = 60
    for target_index in range(1, size):
        left, right = float(lower), float(upper)
        for _ in range(iterations):
            midpoint = (left + right) / 2.0
            if map_axis(midpoint) < target_index:
                left = midpoint
            else:
                right = midpoint
        boundaries.append(right)
    boundaries.append(float(upper))
    return np.asarray(boundaries, dtype=float)


def spatial_grid_boundaries(grid_w=12, grid_h=12):
    """Return x/y bin boundaries implied by the active phi_mapping_grid."""
    x_boundaries = _axis_boundaries(
        lambda x: phi_mapping_grid((x, 0.0), grid_w, grid_h)[0],
        grid_w,
        -1.0,
        1.0,
    )
    y_boundaries = _axis_boundaries(
        lambda y: phi_mapping_grid((0.0, y), grid_w, grid_h)[1],
        grid_h,
        0.0,
        1.5,
    )
    return x_boundaries, y_boundaries


def phi_mapping_sequential(obs, q, grid_w=12, grid_h=12):
    abstract_x, abstract_y = phi_mapping_grid(obs, grid_w, grid_h)
    return abstract_x, abstract_y, q


def lunar_lander_visible_observation_bounds():
    """Return the normalised x/y bounds covered by LunarLander's RGB viewport."""
    # Gymnasium's rendering geometry is constant; keeping the values here lets
    # post-processing run without importing the optional Box2D runtime.
    scale, viewport_width, viewport_height, leg_down = 30.0, 600.0, 400.0, 18.0
    viewport_world_width = viewport_width / scale
    viewport_world_height = viewport_height / scale
    helipad_y = viewport_world_height / 4.0
    lander_y_offset = helipad_y + leg_down / scale
    half_world_height = viewport_world_height / 2.0
    visible_y_min = (0.0 - lander_y_offset) / half_world_height
    visible_y_max = (viewport_world_height - lander_y_offset) / half_world_height
    return -1.0, 1.0, visible_y_min, visible_y_max


def _draw_visible_area_overlay(axis, width, height):
    """Mark which portion of the active abstract grid lies in the RGB viewport."""
    visible_x_min, visible_x_max, visible_y_min, visible_y_max = (
        lunar_lander_visible_observation_bounds()
    )
    x_boundaries, y_boundaries = spatial_grid_boundaries(width, height)

    def to_plot(value, boundaries):
        value = float(np.clip(value, boundaries[0], boundaries[-1]))
        index = int(np.searchsorted(boundaries, value, side="right") - 1)
        index = int(np.clip(index, 0, len(boundaries) - 2))
        lower, upper = boundaries[index], boundaries[index + 1]
        fraction = 0.0 if upper <= lower else (value - lower) / (upper - lower)
        return index - 0.5 + fraction

    left = float(to_plot(visible_x_min, x_boundaries))
    right = float(to_plot(visible_x_max, x_boundaries))
    bottom = float(to_plot(visible_y_min, y_boundaries))
    top = float(to_plot(visible_y_max, y_boundaries))

    axis.add_patch(
        Rectangle(
            (left, bottom),
            right - left,
            top - bottom,
            fill=False,
            edgecolor="#ff1744",
            linewidth=1.4,
            linestyle="--",
            label="Visible RGB viewport",
            zorder=5,
        )
    )
    axis.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.08),
        borderaxespad=0.0,
        frameon=False,
        fontsize=8,
    )

# ==============================
# Abstract-potential heatmaps
# ==============================


def save_sequential_heatmaps(
    abstract_mdp,
    filename_prefix="v_star",
    output_dir=None,
):
    """
    Generates and saves a separate heatmap for V* for each phase defined in the MDP,
    without any waypoint or goal markers (clean heatmap).
    """
    # A caller can isolate each abstraction in img/heatmaps/level1, level2, ...
    output_dir = output_dir or os.path.join("img", "heatmaps")
    os.makedirs(output_dir, exist_ok=True)
    filename_prefix = os.path.basename(filename_prefix)
    
    width, height = abstract_mdp.width, abstract_mdp.height
    
    # Exclude product states that are inconsistent with the proposition label
    # of their current cell. Entering such a cell would already have advanced
    # the DFA, so those states are unreachable in this abstraction.
    def canonical_q(x, y, q):
        truth_assignment = abstract_mdp._get_truth_assignment(x, y)
        return abstract_mdp.automaton.get_next_q(q, truth_assignment)

    for current_q in abstract_mdp.automaton.states:
        matrix = np.full((height, width), np.nan)
        for (x, y, q), value in abstract_mdp.v_star.items():
            if (
                q == current_q
                and 0 <= x < width
                and 0 <= y < height
                and canonical_q(x, y, q) == q
            ):
                matrix[y, x] = value
                
        plt.figure(figsize=(6.4, 5.4), constrained_layout=True)
        # Let matplotlib infer an independent color scale for this DFA state.
        # This exposes the direction of each local gradient instead of
        # compressing it against values from other states or levels.
        im = plt.imshow(matrix, cmap='viridis', origin='lower')
        finite_values = matrix[np.isfinite(matrix)]
        current_vmin = finite_values.min() if len(finite_values) > 0 else 0.0
        current_vmax = finite_values.max() if len(finite_values) > 0 else 0.0
        color_midpoint = (current_vmin + current_vmax) / 2.0
        for y in range(height):
            for x in range(width):
                val = matrix[y, x]
                if np.isnan(val):
                    next_q = canonical_q(x, y, current_q)
                    plt.text(
                        x,
                        y,
                        f"→q{next_q}",
                        ha='center',
                        va='center',
                        color='#d32f2f',
                        fontsize=7,
                        fontweight='bold',
                    )
                    continue
                text_color = 'white' if val < color_midpoint else 'black'
                plt.text(
                    x,
                    y,
                    f"{val:.1f}",
                    ha='center',
                    va='center',
                    color=text_color,
                    fontsize=7,
                )
                    
        plt.colorbar(im, fraction=0.046, pad=0.04, label="Potential Value (V*)")
        
        ax = plt.gca()
        ax.set_xlabel("Grid x")
        ax.set_ylabel("Grid y")
        ax.set_xticks(np.arange(-.5, width, 1), minor=True)
        ax.set_yticks(np.arange(-.5, height, 1), minor=True)
        ax.grid(which='minor', color='w', linestyle='-', linewidth=1, alpha=0.4)
        _draw_visible_area_overlay(ax, width, height)
        
        # Keep the heatmap free of waypoint and goal markers.
            
        plt.savefig(os.path.join(output_dir, f"{filename_prefix}_q{current_q}.png"), dpi=300, bbox_inches='tight')
        plt.close()
        print(
            f" -> Generated V* Heatmap for {abstract_mdp.level_name}, "
            f"DFA State q={current_q}"
        )


def save_multilevel_heatmaps(
    multilevel_mdp,
    filename_prefix="v_star",
    output_root=None,
):
    """Save each level's heatmaps under ``level1``, ``level2``, and so on."""
    output_root = output_root or os.path.join("img", "heatmaps")
    generated_directories = []
    for level_number, abstract_mdp in enumerate(multilevel_mdp.levels, start=1):
        level_directory = os.path.join(output_root, f"level{level_number}")
        save_sequential_heatmaps(
            abstract_mdp,
            filename_prefix=filename_prefix,
            output_dir=level_directory,
        )
        generated_directories.append(level_directory)
    return generated_directories

# ==============================
# Training diagnostics and learning curves
# ==============================


def _prepare_plot_path(filename):
    """Create the destination directory, if one was provided."""
    output_directory = os.path.dirname(os.fspath(filename))
    if output_directory:
        os.makedirs(output_directory, exist_ok=True)


def _style_paper_axis(axis, grid_axis="y"):
    """Apply the unobtrusive axis treatment used by all paper figures."""
    for spine in axis.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.8)
    axis.set_axisbelow(True)
    axis.grid(
        axis=grid_axis,
        color="#d9d9d9",
        linestyle="-",
        linewidth=0.6,
        alpha=0.8,
    )


def _trailing_mean(values, window_size):
    """Return the mean of the current and previous N-1 episodes."""
    return (
        pd.Series(np.asarray(values, dtype=np.float64))
        .rolling(window=window_size, min_periods=1, center=False)
        .mean()
        .to_numpy()
    )


def plot_training_variance(reward_histories, window_size=100, title="Training Performance Across Seeds", filename="img/training_variance.png", label="Learning reward", epsilon_histories=None, epsilon_labels=None):
    """Plot aggregate rewards with variance and optional raw epsilon curves."""
    runs = np.asarray(reward_histories, dtype=np.float64)
    if runs.ndim == 1:
        runs = runs[np.newaxis, :]
    if runs.ndim != 2 or runs.shape[0] == 0 or runs.shape[1] == 0:
        raise ValueError("reward_histories must have shape (num_seeds, episodes)")
    if window_size <= 0:
        raise ValueError("window_size must be greater than zero")

    smoothed_runs = (
        pd.DataFrame(runs.T)
        .rolling(window=window_size, min_periods=1, center=False)
        .mean()
        .to_numpy()
        .T
    )
    mean_reward = np.mean(smoothed_runs, axis=0)
    std_reward = np.std(smoothed_runs, axis=0)
    episodes = np.arange(1, runs.shape[1] + 1)

    _prepare_plot_path(filename)
    fig, ax = plt.subplots(figsize=(7.2, 4.4), constrained_layout=True)
    ax.plot(episodes, mean_reward, color=LEARNING_REWARD_COLOR, linewidth=1.7, label=f"Mean {label}")
    ax.fill_between(
        episodes,
        mean_reward - std_reward,
        mean_reward + std_reward,
        color=LEARNING_REWARD_COLOR,
        alpha=0.18,
        linewidth=0,
        label="±1 SD across seeds",
    )
    ax.set_xlabel("#Episode")
    ax.set_ylabel(label)
    _style_paper_axis(ax)

    legend_handles, legend_labels = ax.get_legend_handles_labels()
    if epsilon_histories is not None:
        epsilon_runs = np.asarray(epsilon_histories, dtype=np.float64)
        if epsilon_runs.ndim == 1:
            epsilon_curves = epsilon_runs[np.newaxis, :]
        elif epsilon_runs.ndim == 2:
            if epsilon_runs.shape[0] == runs.shape[0]:
                epsilon_curves = np.mean(epsilon_runs, axis=0, keepdims=True)
            else:
                epsilon_curves = epsilon_runs
        elif epsilon_runs.ndim == 3:
            epsilon_curves = np.mean(epsilon_runs, axis=0)
        else:
            raise ValueError(
                "epsilon_histories must have shape (episodes), "
                "(seeds, episodes), or (seeds, epsilon_series, episodes)"
            )
        if epsilon_curves.shape[1] != runs.shape[1]:
            raise ValueError("epsilon_histories must contain one value per episode")
        if epsilon_labels is not None and len(epsilon_labels) != len(epsilon_curves):
            raise ValueError("epsilon_labels must match the number of epsilon curves")

        epsilon_axis = ax.twinx()
        for index, epsilon_curve in enumerate(epsilon_curves):
            if epsilon_labels is not None:
                epsilon_label = f"Epsilon q={epsilon_labels[index]}"
            elif len(epsilon_curves) == 1:
                epsilon_label = "Epsilon"
            else:
                epsilon_label = f"Epsilon {index + 1}"
            epsilon_axis.plot(
                episodes,
                epsilon_curve,
                color=EPSILON_COLOR,
                linestyle=EPSILON_LINESTYLES[index % len(EPSILON_LINESTYLES)],
                linewidth=1.4,
                label=epsilon_label,
            )
        epsilon_axis.set_ylabel("Epsilon")
        epsilon_axis.set_ylim(0.0, 1.0)
        epsilon_axis.grid(False)
        epsilon_axis.spines["top"].set_visible(True)
        epsilon_axis.spines["right"].set_visible(True)
        epsilon_handles, epsilon_legend_labels = epsilon_axis.get_legend_handles_labels()
        legend_handles += epsilon_handles
        legend_labels += epsilon_legend_labels

    ax.legend(
        legend_handles,
        legend_labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 1.01),
        ncol=min(len(legend_labels), 4),
        frameon=False,
    )
    fig.savefig(filename, dpi=300, bbox_inches="tight")
    print(f"\n>>> Training variance plot saved to: {filename}")
    plt.close(fig)

def plot_buffer_variance(buffer_histories_runs, window_size=100, filename="img/buffer_variance.png", state_labels=None, title="Replay Buffer Composition Across Seeds"):
    """Plot mean replay-buffer fractions with a ±1 std band across seeds."""
    runs = np.asarray(buffer_histories_runs, dtype=np.float64)
    if runs.ndim == 2:
        runs = runs[np.newaxis, ...]
    if runs.ndim != 3 or 0 in runs.shape:
        raise ValueError(
            "buffer_histories_runs must have shape (num_seeds, num_states, episodes)"
        )
    if window_size <= 0:
        raise ValueError("window_size must be greater than zero")

    smoothed_runs = np.empty_like(runs, dtype=np.float64)
    for seed_index in range(runs.shape[0]):
        for state_index in range(runs.shape[1]):
            smoothed_runs[seed_index, state_index] = (
                pd.Series(runs[seed_index, state_index])
                .rolling(window=window_size, min_periods=1, center=False)
                .mean()
                .to_numpy()
            )

    mean_fractions = np.mean(smoothed_runs, axis=0)
    std_fractions = np.std(smoothed_runs, axis=0)
    episodes = np.arange(1, runs.shape[2] + 1)
    colors = [SERIES_COLORS[index % len(SERIES_COLORS)] for index in range(runs.shape[1])]
    _prepare_plot_path(filename)

    fig, ax = plt.subplots(figsize=(7.2, 4.4), constrained_layout=True)
    for state_index, color in enumerate(colors):
        state_label = state_labels[state_index] if state_labels is not None else state_index
        mean = mean_fractions[state_index]
        std = std_fractions[state_index]
        ax.plot(episodes, mean, color=color, linewidth=1.7, label=f"DFA state q={state_label}")
        ax.fill_between(
            episodes,
            np.clip(mean - std, 0.0, 1.0),
            np.clip(mean + std, 0.0, 1.0),
            color=color,
            alpha=0.16,
            linewidth=0,
        )

    ax.set_xlabel("#Episode")
    ax.set_ylabel("Buffer fraction")
    ax.set_ylim(0, 1.0)
    _style_paper_axis(ax)
    ax.legend(
        loc="lower center",
        bbox_to_anchor=(0.5, 1.01),
        ncol=min(runs.shape[1], 3),
        frameon=False,
    )
    fig.savefig(filename, dpi=300, bbox_inches="tight")
    print(f"\n>>> Buffer variance plot saved to: {filename}")
    plt.close(fig)

def plot_buffer_fractions(buffer_histories, window_size=100, filename="img/buffer_fractions.png", state_labels=None, title="Replay Buffer Composition"):
    """
    Plots the replay buffer composition for N phases dynamically.
    """
    _prepare_plot_path(filename)
    fig, ax = plt.subplots(figsize=(7.2, 4.4), constrained_layout=True)
    x_axis = np.arange(1, len(buffer_histories[0]) + 1)
    
    colors = [SERIES_COLORS[index % len(SERIES_COLORS)] for index in range(len(buffer_histories))]
    for idx, history in enumerate(buffer_histories):
        ma = _trailing_mean(history, window_size)
        state_label = state_labels[idx] if state_labels is not None else idx
        ax.plot(x_axis, ma, color=colors[idx], linewidth=1.7, label=f'DFA state q={state_label}')
    
    ax.set_xlabel("#Episode")
    ax.set_ylabel("Buffer fraction")
    ax.set_ylim(0, 1.0)
    _style_paper_axis(ax)
    ax.legend(
        loc="lower center",
        bbox_to_anchor=(0.5, 1.01),
        ncol=min(len(buffer_histories), 3),
        frameon=False,
    )
    fig.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close(fig)

def plot_shaping_reward_breakdown(true_rewards, total_rewards, eps_histories, window_size=100, filename="img/shaping_reward_breakdown.png", title="Shaping Agent Reward Analysis"):
    """Plot rewards and epsilon together using independent, well-defined axes."""
    if window_size <= 0:
        raise ValueError("window_size must be greater than zero")
    true_rewards = np.asarray(true_rewards, dtype=np.float64)
    total_rewards = np.asarray(total_rewards, dtype=np.float64)
    if true_rewards.ndim != 1 or total_rewards.shape != true_rewards.shape:
        raise ValueError("true_rewards and total_rewards must be equally sized vectors")

    exploration_runs = np.asarray(eps_histories, dtype=np.float64)
    if exploration_runs.ndim == 1:
        exploration_runs = exploration_runs[np.newaxis, :]
    elif exploration_runs.ndim == 2 and exploration_runs.shape[1] != len(true_rewards):
        if exploration_runs.shape[0] == len(true_rewards):
            exploration_runs = exploration_runs.T
    if exploration_runs.ndim != 2 or exploration_runs.shape[1] != len(true_rewards):
        raise ValueError("eps_histories must contain one value per episode")

    _prepare_plot_path(filename)
    figure, reward_axis = plt.subplots(
        figsize=(7.2, 4.4),
        constrained_layout=True,
    )
    epsilon_axis = reward_axis.twinx()
    episodes = np.arange(1, len(true_rewards) + 1)
    reward_axis.plot(
        episodes,
        _trailing_mean(true_rewards, window_size),
        color=TASK_REWARD_COLOR,
        linewidth=1.6,
        label="Task reward",
    )
    reward_axis.plot(
        episodes,
        _trailing_mean(total_rewards, window_size),
        color=LEARNING_REWARD_COLOR,
        linewidth=1.7,
        label="Learning reward (goal + shaping)",
    )
    reward_axis.axhline(0.0, color=REFERENCE_LINE_COLOR, linewidth=0.7, alpha=0.7)
    reward_axis.set_xlabel("#Episode")
    reward_axis.set_ylabel("Episode Reward")
    _style_paper_axis(reward_axis)

    for index, history in enumerate(exploration_runs):
        phase = "Goal" if index == len(exploration_runs) - 1 else f"WP {index + 1}"
        label = "Epsilon" if len(exploration_runs) == 1 else f"Epsilon q={index} ({phase})"
        epsilon_axis.plot(
            episodes,
            history,
            color=EPSILON_COLOR,
            linestyle=EPSILON_LINESTYLES[index % len(EPSILON_LINESTYLES)],
            linewidth=1.4,
            label=label,
        )
    epsilon_axis.set_ylabel("Epsilon")
    epsilon_axis.set_ylim(0.0, 1.0)
    epsilon_axis.grid(False)
    epsilon_axis.spines["top"].set_visible(True)
    epsilon_axis.spines["right"].set_visible(True)

    reward_lines, reward_labels = reward_axis.get_legend_handles_labels()
    epsilon_lines, epsilon_labels = epsilon_axis.get_legend_handles_labels()
    reward_axis.legend(
        reward_lines + epsilon_lines,
        reward_labels + epsilon_labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 1.01),
        ncol=min(len(reward_labels) + len(epsilon_labels), 4),
        frameon=False,
    )
    figure.savefig(filename, dpi=300, bbox_inches="tight")
    plt.close(figure)


## 7. Write the automaton validator

In [ ]:
%%writefile automaton_validator.py
# ==============================
# Standard library imports
# ==============================

import itertools
import re
from collections import deque


# ==============================
# Formula and valuation helpers
# ==============================

LTLF_OPERATORS = {"F", "G", "M", "R", "U", "W", "X", "false", "true"}


def _extract_formula_propositions(formula):
    """Extract atomic proposition names from an LTLf formula."""
    tokens = set(re.findall(r"[A-Za-z_][A-Za-z0-9_]*", formula))
    return sorted(token for token in tokens if token not in LTLF_OPERATORS)


def _generate_truth_assignments(propositions):
    """Generate every Boolean valuation for the formula propositions."""
    for values in itertools.product((False, True), repeat=len(propositions)):
        yield dict(zip(propositions, values))


def _matching_transitions(automaton, state, truth_assignment):
    """Return all outgoing DFA transitions enabled by one truth assignment."""
    return [(guard, destination) for guard, destination in automaton.transitions.get(state, []) if automaton._eval_guard(guard, truth_assignment)]


# ==============================
# Validation report
# ==============================

class AutomatonValidationReport:
    """Collect validation errors, warnings, and useful DFA statistics."""

    def __init__(self, formula, propositions):
        self.formula = formula
        self.propositions = propositions
        self.errors = []
        self.warnings = []
        self.statistics = {}

    @property
    def is_valid(self):
        """Return whether validation completed without errors."""
        return not self.errors

    def add_error(self, message):
        """Append one blocking validation error."""
        self.errors.append(message)

    def add_warning(self, message):
        """Append one non-blocking validation warning."""
        self.warnings.append(message)

    def format(self):
        """Format the complete validation report for console and log output."""
        status = "VALID" if self.is_valid else "INVALID"
        lines = [
            "=== AUTOMATON VALIDATION ===",
            f"Status: {status}",
            f"Formula propositions: {self.propositions}",
            f"Statistics: {self.statistics}",
        ]
        if self.errors:
            lines.append("Errors:")
            lines.extend(f"- {message}" for message in self.errors)
        if self.warnings:
            lines.append("Warnings:")
            lines.extend(f"- {message}" for message in self.warnings)
        return "\n".join(lines)

    def raise_if_invalid(self):
        """Raise an exception containing the report when validation fails."""
        if not self.is_valid:
            raise ValueError(self.format())


# ==============================
# DFA validation
# ==============================

def validate_automaton(automaton, waypoints_dict, width=None, height=None, max_propositions=12, raise_on_error=True):
    """Validate DFA structure, guards, reachability, propositions, and waypoint coordinates."""
    propositions = _extract_formula_propositions(automaton.formula_str)
    report = AutomatonValidationReport(automaton.formula_str, propositions)
    states = set(automaton.states)
    waypoint_propositions = set(waypoints_dict)

    # Validate formula propositions and waypoint declarations.
    missing_waypoints = sorted(set(propositions) - waypoint_propositions)
    unused_waypoints = sorted(waypoint_propositions - set(propositions))
    if missing_waypoints:
        report.add_error(f"Formula propositions without coordinates: {missing_waypoints}")
    if unused_waypoints:
        report.add_warning(f"Waypoint propositions not used by the formula: {unused_waypoints}")
    if len(propositions) > max_propositions:
        report.add_error(f"The formula has {len(propositions)} propositions; exhaustive validation is limited to {max_propositions}")

    # Validate waypoint coordinate structure and grid bounds.
    for proposition, coordinates in waypoints_dict.items():
        if not isinstance(coordinates, (tuple, list)) or len(coordinates) != 2:
            report.add_error(f"Waypoint {proposition!r} must contain exactly two coordinates")
            continue
        x, y = coordinates
        if not isinstance(x, int) or not isinstance(y, int):
            report.add_error(f"Waypoint {proposition!r} coordinates must be integers")
        if width is not None and not 0 <= x < width:
            report.add_error(f"Waypoint {proposition!r} has x={x}, outside [0, {width - 1}]")
        if height is not None and not 0 <= y < height:
            report.add_error(f"Waypoint {proposition!r} has y={y}, outside [0, {height - 1}]")

    # Validate initial, accepting, source, and destination states.
    if not states:
        report.add_error("The DFA contains no states")
    if automaton.get_initial_q() not in states:
        report.add_error(f"Initial state {automaton.get_initial_q()!r} is not part of the DFA")
    unknown_accepting = sorted(set(automaton.accepting_states) - states)
    if unknown_accepting:
        report.add_error(f"Unknown accepting states: {unknown_accepting}")
    if not automaton.accepting_states:
        report.add_error("The DFA has no accepting states")
    for source, transitions in automaton.transitions.items():
        if source not in states:
            report.add_error(f"Transition source {source!r} is not part of the DFA")
        for guard, destination in transitions:
            if destination not in states:
                report.add_error(f"Transition {source!r} --[{guard}]--> {destination!r} targets an unknown state")

    # Validate deterministic and complete behavior over the full Boolean alphabet.
    truth_assignments = list(_generate_truth_assignments(propositions)) if len(propositions) <= max_propositions else []
    reachable_states = {automaton.get_initial_q()} if automaton.get_initial_q() in states else set()
    frontier = deque(reachable_states)
    checked_pairs = 0
    ambiguous_pairs = 0
    incomplete_pairs = 0

    for state in states:
        for truth_assignment in truth_assignments:
            matches = _matching_transitions(automaton, state, truth_assignment)
            checked_pairs += 1
            if not matches:
                incomplete_pairs += 1
                report.add_error(f"No transition from state {state!r} for valuation {truth_assignment}")
            elif len(matches) > 1:
                ambiguous_pairs += 1
                guards = [guard for guard, _ in matches]
                report.add_error(f"Ambiguous transitions from state {state!r} for valuation {truth_assignment}: {guards}")
            else:
                expected_destination = matches[0][1]
                actual_destination = automaton.get_next_q(state, truth_assignment)
                if actual_destination != expected_destination:
                    report.add_error(f"get_next_q returned {actual_destination!r}, expected {expected_destination!r} from state {state!r}")

    # Traverse the validated transition relation to find unreachable states.
    while frontier and truth_assignments:
        state = frontier.popleft()
        for truth_assignment in truth_assignments:
            matches = _matching_transitions(automaton, state, truth_assignment)
            if len(matches) == 1 and matches[0][1] not in reachable_states:
                reachable_states.add(matches[0][1])
                frontier.append(matches[0][1])

    unreachable_states = sorted(states - reachable_states)
    unreachable_accepting = sorted(set(automaton.accepting_states) - reachable_states)
    if unreachable_states:
        report.add_warning(f"Unreachable DFA states: {unreachable_states}")
    if unreachable_accepting:
        report.add_error(f"No accepting path reaches states: {unreachable_accepting}")

    # Store compact statistics for diagnostics and reproducibility.
    report.statistics = {
        "states": len(states),
        "accepting_states": len(automaton.accepting_states),
        "transitions": sum(len(transitions) for transitions in automaton.transitions.values()),
        "valuations": len(truth_assignments),
        "state_valuation_pairs": checked_pairs,
        "ambiguous_pairs": ambiguous_pairs,
        "incomplete_pairs": incomplete_pairs,
        "reachable_states": len(reachable_states),
    }

    if raise_on_error:
        report.raise_if_invalid()
    return report


## 8. Write the training program

In [ ]:
%%writefile trainer.py
# ==============================
# Standard library imports
# ==============================

import argparse
import json
import os
import random
import re
import shutil
from collections import Counter
from pathlib import Path

# ==============================
# External and project imports
# ==============================

import gymnasium as gym
import numpy as np
import torch

from abstraction import AbstractionConfig
from abstract_mdps import LTLfAutomaton, MultiLevelWaypointMDP
from agent import HierarchicalDQNLearner
from automaton_validator import validate_automaton
from utils import (
    phi_mapping_sequential,
    plot_buffer_fractions,
    plot_buffer_variance,
    plot_shaping_reward_breakdown,
    plot_training_variance,
    save_multilevel_heatmaps,
)

SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
FRAMEWORK_DIR = (
    os.path.dirname(SCRIPT_DIR)
    if os.path.basename(SCRIPT_DIR) == "src"
    else SCRIPT_DIR
)


MULTI_EPSILON_MIN = 0.08
MULTI_EPSILON_UNLOCK_THRESHOLD = 0.081
EPSILON_STRATEGIES = {"cascade", "visited"}


# ==============================
# Data and state helpers
# ==============================

def _positive_int(value):
    """Parse a strictly positive command-line integer."""
    number = int(value)
    if number <= 0:
        raise argparse.ArgumentTypeError("must be greater than zero")
    return number


def _experiment_name(value):
    """Validate a safe single-directory experiment name."""
    name = str(value).strip()
    if len(name) > 100 or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]*", name):
        raise argparse.ArgumentTypeError(
            "must start with a letter or digit and contain only letters, digits, '.', '_' or '-'"
        )
    return name


def _resolve_config_path(requested_path, default_filename, experiment_dir, post_process):
    """Resolve a config, preferring the experiment snapshot during post-processing."""
    requested = Path(requested_path).expanduser()
    framework_default = Path(SCRIPT_DIR) / default_filename
    uses_default = (
        str(requested_path) == default_filename
        or requested.resolve() == framework_default.resolve()
    )
    candidates = []
    if post_process and uses_default:
        candidates.extend(
            [
                Path(experiment_dir) / default_filename,
                Path(experiment_dir) / "results" / default_filename,
            ]
        )
    candidates.append(requested)
    if not requested.is_absolute():
        candidates.append(Path(SCRIPT_DIR) / requested)

    for candidate in candidates:
        if candidate.is_file():
            return candidate
    checked = "\n  - ".join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(f"Configuration file not found. Checked:\n  - {checked}")


def _archive_config(config_path, experiment_dir, filename):
    """Store the exact training configuration beside the experiment outputs."""
    destination = Path(experiment_dir) / filename
    if Path(config_path).resolve() != destination.resolve():
        shutil.copy2(config_path, destination)


def _resolve_metrics_path(experiment_dir, filename):
    """Find metrics in the results subfolder or the legacy experiment root."""
    candidates = [
        Path(experiment_dir) / "results" / filename,
        Path(experiment_dir) / filename,
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    checked = "\n  - ".join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(f"Training data not found. Checked:\n  - {checked}")


def _organize_policy_files(policy_dir):
    """Move checkpoints from legacy layouts into policy/best and policy/last."""
    policy_root = Path(policy_dir)
    destinations = {
        "best": policy_root / "best",
        "last": policy_root / "last",
    }
    for destination in destinations.values():
        destination.mkdir(parents=True, exist_ok=True)

    for category, destination in destinations.items():
        for source in policy_root.glob(f"{category}_policy*"):
            if source.is_file() and not (destination / source.name).exists():
                shutil.move(str(source), destination / source.name)


def _organize_legacy_seed_plots(image_dir):
    """Move legacy per-seed plots from img/ into img/seed_<seed>/ folders."""
    pattern = re.compile(r"^((?:reward_breakdown|buffer_fractions)_.+)_seed_(-?\d+)\.png$")
    for source in Path(image_dir).glob("*.png"):
        match = pattern.fullmatch(source.name)
        if not match:
            continue
        destination_dir = Path(image_dir) / f"seed_{match.group(2)}"
        destination_dir.mkdir(parents=True, exist_ok=True)
        destination = destination_dir / f"{match.group(1)}.png"
        if not destination.exists():
            shutil.move(str(source), destination)


def save_training_data(filename, **kwargs):
    """Convert training metrics to arrays and save them in a compressed NPZ file."""
    # Preserve numeric dtypes and rectangular shapes for direct plotting.
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    np_data = {key: np.asarray(value) for key, value in kwargs.items()}
    if any(array.dtype == object for array in np_data.values()):
        raise ValueError("Training metrics must be rectangular numeric arrays")
    np.savez_compressed(filename, **np_data)
    print(f"\nTraining data saved to: {filename}")


def _set_training_seed(seed, env=None):
    """Seed every random generator used by DDQN and LunarLander."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if env is not None:
        env.action_space.seed(seed)


def _aggregate_seed_metrics(seed_metrics, seeds):
    """Keep first-run compatibility and add every metric stacked by seed."""
    if not seed_metrics:
        raise ValueError("At least one seed run is required")
    aggregated = dict(seed_metrics[0])
    aggregated["seeds"] = np.asarray(seeds, dtype=np.int64)
    for key in seed_metrics[0]:
        if key == "automaton_states":
            continue
        try:
            aggregated[f"{key}_runs"] = np.stack(
                [np.asarray(metrics[key]) for metrics in seed_metrics]
            )
        except ValueError as error:
            raise ValueError(f"Metric {key!r} has inconsistent shapes across seeds") from error
    for key in ("task_rewards", "learning_rewards", "shaping_rewards"):
        runs = aggregated[f"{key}_runs"]
        aggregated[f"{key}_mean"] = np.mean(runs, axis=0)
        aggregated[f"{key}_variance"] = np.var(runs, axis=0)
    return aggregated


def _abstract_position(observation, abstract_mdp):
    """Map a raw environment observation to its abstract spatial coordinates."""
    x, y, _ = phi_mapping_sequential(
        observation, 0, abstract_mdp.width, abstract_mdp.height
    )
    return x, y


def _augment_state(observation, q, state_to_index):
    """Append the DFA one-hot used by replay accounting and head routing."""
    one_hot = np.zeros(len(state_to_index), dtype=np.float32)
    one_hot[state_to_index[q]] = 1.0
    return np.concatenate((observation, one_hot)).astype(np.float32)


def _evaluate_initial_automaton_state(observation, abstract_mdp):
    """Consume the initial observation from the DFA pre-trace state and return the first active state."""
    initial_x, initial_y = _abstract_position(observation, abstract_mdp)
    initial_truth_assignment = abstract_mdp._get_truth_assignment(initial_x, initial_y)
    pre_trace_q = abstract_mdp.automaton.get_initial_q()
    return abstract_mdp.automaton.get_next_q(pre_trace_q, initial_truth_assignment)


def _format_counter(counter):
    """Convert a DFA transition counter into a compact human-readable string."""
    if not counter:
        return "none"
    return ", ".join(f"{source}->{destination}: {count}" for (source, destination), count in sorted(counter.items()))


def _decay_multi_epsilons(epsilons, reached_phase_this_episode, eps_decay):
    """Apply the cascaded per-DFA-state epsilon decay used by the backup trainer."""
    if len(epsilons) != len(reached_phase_this_episode):
        raise ValueError("epsilons and reached-phase flags must have the same length")
    if not epsilons:
        raise ValueError("at least one epsilon is required")

    next_epsilons = list(epsilons)
    next_epsilons[0] = max(
        MULTI_EPSILON_MIN,
        next_epsilons[0] * eps_decay,
    )
    for index in range(1, len(next_epsilons)):
        predecessor_is_unlocked = (
            reached_phase_this_episode[index - 1]
            and next_epsilons[index - 1]
            <= MULTI_EPSILON_UNLOCK_THRESHOLD
        )
        if predecessor_is_unlocked:
            next_epsilons[index] = max(
                MULTI_EPSILON_MIN,
                next_epsilons[index] * eps_decay,
            )
    return next_epsilons


def _decay_visited_epsilons(epsilons, acted_state_indices, eps_decay):
    """Decay once each epsilon whose DFA policy acted during the episode."""
    next_epsilons = list(epsilons)
    for index in set(acted_state_indices):
        if not 0 <= index < len(next_epsilons):
            raise IndexError("acted DFA-state index is out of range")
        next_epsilons[index] = max(
            MULTI_EPSILON_MIN,
            next_epsilons[index] * eps_decay,
        )
    return next_epsilons


# ==============================
# Logging and checkpoint helpers
# ==============================

def _write_log(message, log_handle=None):
    """Print a message and optionally append it to the active log file."""
    print(message)
    if log_handle:
        log_handle.write(message)
        log_handle.flush()


def _write_run_header(log_handle, episodes, use_shaping, K, goal_reward, abstract_mdp, automaton_states, agent, epsilon_strategy, training_shaping_gamma):
    """Write the configuration and DFA metadata at the beginning of a training run."""
    if not log_handle:
        return
    automaton = abstract_mdp.automaton
    output_heads = (
        len(automaton_states)
        if agent.network_architecture == "multi-head"
        else 1
    )
    shaping_formula = (
        "K*(gamma*Phi(next)-Phi(state))"
        if training_shaping_gamma
        else "K*(Phi(next)-Phi(state))"
    )
    header = (
        "\n=== NEW MULTI-EPSILON RUN ===\n"
        f"episodes={episodes}, shaping={use_shaping}, K={K}, goal_reward={goal_reward}, gamma={abstract_mdp.gamma}\n"
        f"training_shaping_gamma={training_shaping_gamma}, shaping_formula={shaping_formula}\n"
        f"network={agent.algo_name}, output_heads={output_heads}\n"
        f"epsilon_strategy={epsilon_strategy}\n"
        f"epsilon_min={MULTI_EPSILON_MIN}, epsilon_unlock_threshold={MULTI_EPSILON_UNLOCK_THRESHOLD}\n"
        f"inter_level_shaping={abstract_mdp.upper_level_mdp is not None}, "
        f"inter_level_K={abstract_mdp.inter_level_shaping_scale}\n"
        f"formula={automaton.formula_str}\n"
        f"waypoints={abstract_mdp.waypoints_dict}\n"
        f"dfa_states={automaton_states}, pre_trace={automaton.get_initial_q()}, accepting={sorted(automaton.accepting_states)}\n"
    )
    log_handle.write(header)
    log_handle.flush()


def _should_log(episode, episodes, log_interval):
    """Return whether the current episode requires a periodic training report."""
    return episode == 0 or episode + 1 == episodes or (episode + 1) % log_interval == 0


def _build_training_log(episode, episodes, log_interval, automaton_states, agent, histories, cumulative_counters):
    """Build a report containing recent metrics and cumulative DFA counters."""
    window = min(log_interval, episode + 1)
    recent_slice = slice(-window, None)
    recent_transitions = Counter()
    for transitions in histories["transition_counters"][-window:]:
        recent_transitions.update(transitions)

    recent_state_visits = np.asarray(histories["state_visits"], dtype=np.int64)[:, -window:].sum(axis=1)
    recent_state_entries = np.asarray(histories["state_entries"], dtype=np.int64)[:, -window:].sum(axis=1)
    buffer_details = ", ".join(f"{q}: {agent.memory.q_fraction_onehot(index, len(automaton_states)):.1%}" for index, q in enumerate(automaton_states))
    recent_visits_details = ", ".join(f"{q}: {recent_state_visits[index]}" for index, q in enumerate(automaton_states))
    recent_entries_details = ", ".join(f"{q}: {recent_state_entries[index]}" for index, q in enumerate(automaton_states))
    cumulative_visits_details = ", ".join(f"{q}: {cumulative_counters['state_visits'][q]}" for q in automaton_states)
    cumulative_entries_details = ", ".join(f"{q}: {cumulative_counters['state_entries'][q]}" for q in automaton_states)
    epsilon_details = ", ".join(
        f"{q}: {histories['epsilons'][index][-1]:.5f}"
        for index, q in enumerate(automaton_states)
    )

    return (
        "\n"
        f"[Episode {episode + 1}/{episodes} | last {window}]\n"
        f"success rate                : {np.mean(histories['successes'][recent_slice]):.1%} (cumulative {np.mean(histories['successes']):.1%})\n"
        f"synthetic task reward       : {np.mean(histories['task_rewards'][recent_slice]):.3f}\n"
        f"shaping reward              : {np.mean(histories['shaping_rewards'][recent_slice]):.3f}\n"
        f"learning reward             : {np.mean(histories['learning_rewards'][recent_slice]):.3f}\n"
        f"episode length              : {np.mean(histories['episode_lengths'][recent_slice]):.1f}\n"
        f"abstract changes / episode  : {np.mean(histories['abstract_changes'][recent_slice]):.1f}\n"
        f"DFA transitions / episode   : {np.mean(histories['dfa_transitions'][recent_slice]):.2f}\n"
        f"DFA transitions in window   : {_format_counter(recent_transitions)}\n"
        f"epsilons (next episode)      : {epsilon_details}\n"
        f"replay buffer                : {len(agent.memory)} samples [{buffer_details}]\n"
        f"DFA state visits in window   : {recent_visits_details}\n"
        f"DFA state visits cumulative  : {cumulative_visits_details}\n"
        f"DFA state entries in window  : {recent_entries_details}\n"
        f"DFA state entries cumulative : {cumulative_entries_details}\n"
        f"transitions cumulative       : {_format_counter(cumulative_counters['transitions'])}\n"
        f"accepted directly from s0    : {cumulative_counters['initial_acceptances']}\n"
        f"Gym endings cumulative       : terminated={cumulative_counters['env_terminated']}, truncated={cumulative_counters['env_truncated']}\n"
    )


def _save_named_policy(agent, policy_name):
    """Save the current policy using a stable descriptive filename."""
    category = "best" if policy_name.startswith("best_policy") else "last"
    os.makedirs(os.path.join(agent.policy_dir, category), exist_ok=True)
    agent.policy_name = os.path.join(category, policy_name)
    agent._save_policy()


def _monitoring_average(values, episode, log_interval):
    """Return the mean over the active monitoring window."""
    window = min(log_interval, episode + 1)
    return float(np.mean(values[-window:]))


def _validate_training_setup(automaton, state_to_index, episodes, log_interval):
    """Validate DFA consistency and the numeric parameters required by training."""
    if automaton.get_initial_q() not in state_to_index:
        raise ValueError("The DFA initial state is missing from automaton.states")
    if not automaton.accepting_states.issubset(state_to_index):
        raise ValueError("At least one accepting DFA state is missing from automaton.states")
    if episodes <= 0:
        raise ValueError("episodes must be greater than zero")
    if log_interval <= 0:
        raise ValueError("log_interval must be greater than zero")


def _build_training_results(histories, initial_acceptance_history, buffer_histories, automaton_states, best_mean_reward, best_policy_episode):
    """Select and name the numeric histories returned by the training loop."""
    return {
        "task_rewards": histories["task_rewards"],
        "learning_rewards": histories["learning_rewards"],
        "shaping_rewards": histories["shaping_rewards"],
        "epsilon_history": histories["epsilons"],
        "buffer_histories": buffer_histories,
        "state_visit_histories": histories["state_visits"],
        "state_entry_histories": histories["state_entries"],
        "successes": histories["successes"],
        "initial_acceptances": initial_acceptance_history,
        "episode_lengths": histories["episode_lengths"],
        "abstract_changes": histories["abstract_changes"],
        "dfa_transitions": histories["dfa_transitions"],
        "automaton_states": automaton_states,
        "best_mean_learning_reward": best_mean_reward,
        "best_policy_episode": best_policy_episode,
    }


# ==============================
# Training loop
# ==============================

def run_sequential_training(env, agent, abstract_mdp, episodes, goal_reward=10000, save_policy=True, use_shaping=True, K=1.0, log_file=None, log_interval=100, epsilon_strategy="cascade", training_shaping_gamma=True, seed=None, policy_suffix=""):
    """
    Train DDQN with one epsilon per DFA state and a selectable decay strategy.

    The Gym reward is deliberately discarded. The learning reward is the
    synthetic goal reward plus potential-based shaping. Shaping is evaluated
    only when the complete abstract state (x, y, q) changes.
    """
    # Build a stable mapping between DFA states and neural-network features.
    automaton = abstract_mdp.automaton
    automaton_states = list(automaton.states)
    state_to_index = {q: index for index, q in enumerate(automaton_states)}
    num_states = len(automaton_states)
    epsilons = [agent.eps] * num_states
    if epsilon_strategy not in EPSILON_STRATEGIES:
        raise ValueError(
            f"epsilon_strategy must be one of: {sorted(EPSILON_STRATEGIES)}"
        )

    # Fail early if the DFA or training parameters are inconsistent.
    _validate_training_setup(automaton, state_to_index, episodes, log_interval)

    # Store episode-level metrics for plots and post-processing.
    task_reward_history = []
    learning_reward_history = []
    shaping_reward_history = []
    epsilon_histories = [[] for _ in automaton_states]
    episode_length_history = []
    success_history = []
    initial_acceptance_history = []
    abstract_change_history = []
    dfa_transition_history = []
    transition_counter_history = []
    buffer_histories = [[] for _ in automaton_states]
    state_visit_histories = [[] for _ in automaton_states]
    state_entry_histories = [[] for _ in automaton_states]
    histories = {
        "task_rewards": task_reward_history,
        "learning_rewards": learning_reward_history,
        "shaping_rewards": shaping_reward_history,
        "epsilons": epsilon_histories,
        "episode_lengths": episode_length_history,
        "successes": success_history,
        "abstract_changes": abstract_change_history,
        "dfa_transitions": dfa_transition_history,
        "transition_counters": transition_counter_history,
        "state_visits": state_visit_histories,
        "state_entries": state_entry_histories,
    }

    # Keep cumulative counters for diagnostics shown during training.
    cumulative_state_visits = Counter()
    cumulative_state_entries = Counter()
    cumulative_transitions = Counter()
    cumulative_env_terminated = 0
    cumulative_env_truncated = 0
    cumulative_initial_acceptances = 0
    best_mean_reward = -np.inf
    best_policy_episode = 0

    # Open one append-only log file for the complete run.
    log_handle = open(log_file, "a", encoding="utf-8") if log_file else None
    _write_run_header(log_handle, episodes, use_shaping, K, goal_reward, abstract_mdp, automaton_states, agent, epsilon_strategy, training_shaping_gamma)

    try:
        for episode in range(episodes):
            # Reset the environment and consume s0 before selecting the first action.
            raw_state, _ = env.reset(seed=seed if episode == 0 else None)
            q = _evaluate_initial_automaton_state(raw_state, abstract_mdp)
            if q not in state_to_index:
                raise RuntimeError(f"DFA returned unknown initial state {q!r} after evaluating s0")
            augmented_state = _augment_state(raw_state, q, state_to_index)

            # Unlock later epsilon schedules only after their predecessor has
            # been reached during the current episode.
            reached_phase_this_episode = [False] * num_states
            acted_state_indices = set()

            # Reset counters local to the current episode.
            succeeded = automaton.is_goal_reached(q)
            episode_done = succeeded
            episode_steps = 0
            episode_task_reward = float(goal_reward) if succeeded else 0.0
            episode_shaping_reward = 0.0
            episode_abstract_changes = 0
            episode_dfa_transitions = 0
            episode_state_visits = [0] * num_states
            episode_state_visits[state_to_index[q]] = 1
            # Count s0 as an entry from the virtual pre-trace state.
            episode_state_entries = [0] * num_states
            episode_state_entries[state_to_index[q]] = 1
            episode_transitions = Counter()
            if succeeded:
                cumulative_initial_acceptances += 1
            cumulative_state_visits[q] += 1
            cumulative_state_entries[q] += 1

            while not episode_done:
                # Select an action using the epsilon associated with the
                # current DFA state.
                current_q_index = state_to_index[q]
                agent.eps = epsilons[current_q_index]
                acted_state_indices.add(current_q_index)
                action = agent.select_action(augmented_state)

                # The environment reward is intentionally not part of training.
                next_raw_state, _ignored_env_reward, env_terminated, env_truncated, _ = env.step(action)

                # Map the transition to abstract spatial states.
                x, y = _abstract_position(raw_state, abstract_mdp)
                next_x, next_y = _abstract_position(next_raw_state, abstract_mdp)
                abstract_state = (x, y, q)

                # Advance the DFA using propositions true in the arrival state.
                truth_assignment = abstract_mdp._get_truth_assignment(next_x, next_y)
                next_q = automaton.get_next_q(q, truth_assignment)
                if next_q not in state_to_index:
                    raise RuntimeError(f"DFA returned unknown state {next_q!r} from state {q!r}")

                # Count every arrival in a DFA state, including self-transitions.
                episode_state_visits[state_to_index[next_q]] += 1
                cumulative_state_visits[next_q] += 1

                # Track physical abstraction changes separately from DFA changes.
                abstract_next_state = (next_x, next_y, next_q)
                abstract_changed = abstract_state != abstract_next_state
                dfa_changed = next_q != q

                if abstract_changed:
                    episode_abstract_changes += 1
                if dfa_changed:
                    transition = (q, next_q)
                    episode_dfa_transitions += 1
                    episode_state_entries[state_to_index[next_q]] += 1
                    episode_transitions[transition] += 1
                    cumulative_state_entries[next_q] += 1
                    cumulative_transitions[transition] += 1
                    if not automaton.is_goal_reached(next_q):
                        reached_phase_this_episode[state_to_index[q]] = True

                # Assign the synthetic task reward only on DFA acceptance.
                synthetic_goal_reward = 0.0
                if automaton.is_goal_reached(next_q):
                    synthetic_goal_reward = float(goal_reward)
                    succeeded = True

                # Stop data collection on any Gym ending or DFA success.
                # A truncation (for example Gym's time limit) ends data
                # collection, but it is not an MDP terminal state: DDQN must
                # still bootstrap from its final observation.
                episode_done = env_terminated or env_truncated or succeeded
                bootstrap_terminal = env_terminated or succeeded
                next_augmented_state = _augment_state(next_raw_state, next_q, state_to_index)

                # Evaluate shaping only when the complete abstract state changes.
                shaping_signal = 0.0
                if use_shaping and abstract_changed:
                    phi_state = abstract_mdp.v_star.get(abstract_state, 0.0)
                    phi_next_state = abstract_mdp.v_star.get(abstract_next_state, 0.0)
                    training_discount = abstract_mdp.gamma if training_shaping_gamma else 1.0
                    shaping_signal = K * (training_discount * phi_next_state - phi_state)

                # Store the transition and perform one DDQN optimization step.
                learning_reward = synthetic_goal_reward + shaping_signal
                agent.memory.push(
                    augmented_state,
                    action,
                    learning_reward,
                    next_augmented_state,
                    bootstrap_terminal,
                )
                agent.optimize_model()

                # Update the episode totals and move to the next state.
                episode_steps += 1
                episode_task_reward += synthetic_goal_reward
                episode_shaping_reward += shaping_signal
                raw_state = next_raw_state
                augmented_state = next_augmented_state
                q = next_q

                # Count Gym endings for diagnostics without using its reward.
                if env_terminated:
                    cumulative_env_terminated += 1
                if env_truncated:
                    cumulative_env_truncated += 1

            if epsilon_strategy == "cascade":
                # The first epsilon always decays. Each later schedule starts
                # only after the previous phase is reached while its epsilon
                # is already at the minimum.
                epsilons = _decay_multi_epsilons(
                    epsilons,
                    reached_phase_this_episode,
                    agent.eps_decay,
                )
            else:
                # Decay once per episode every epsilon whose DFA policy was
                # used for at least one action, independently of the others.
                epsilons = _decay_visited_epsilons(
                    epsilons,
                    acted_state_indices,
                    agent.eps_decay,
                )

            # Save the metrics collected for this episode.
            episode_learning_reward = episode_task_reward + episode_shaping_reward
            task_reward_history.append(episode_task_reward)
            shaping_reward_history.append(episode_shaping_reward)
            learning_reward_history.append(episode_learning_reward)
            episode_length_history.append(episode_steps)
            success_history.append(int(succeeded))
            initial_acceptance_history.append(int(episode_steps == 0 and succeeded))
            abstract_change_history.append(episode_abstract_changes)
            dfa_transition_history.append(episode_dfa_transitions)
            transition_counter_history.append(episode_transitions)

            # Record replay-buffer composition, state visits, and entries from other states.
            for index in range(num_states):
                epsilon_histories[index].append(epsilons[index])
                buffer_histories[index].append(agent.memory.q_fraction_onehot(index, num_states))
                state_visit_histories[index].append(episode_state_visits[index])
                state_entry_histories[index].append(episode_state_entries[index])

            # Print recent and cumulative diagnostics at the requested interval.
            if _should_log(episode, episodes, log_interval):
                cumulative_counters = {"state_visits": cumulative_state_visits, "state_entries": cumulative_state_entries, "transitions": cumulative_transitions, "initial_acceptances": cumulative_initial_acceptances, "env_terminated": cumulative_env_terminated, "env_truncated": cumulative_env_truncated}
                _write_log(_build_training_log(episode, episodes, log_interval, automaton_states, agent, histories, cumulative_counters), log_handle)

                # Replace the best policy when the monitored mean reward improves.
                monitored_mean_reward = _monitoring_average(learning_reward_history, episode, log_interval)
                if monitored_mean_reward > best_mean_reward:
                    best_mean_reward = monitored_mean_reward
                    best_policy_episode = episode + 1
                    if save_policy:
                        _save_named_policy(agent, f"best_policy{policy_suffix}.pth")
                        _write_log(f"Best policy updated at episode {best_policy_episode}: mean learning reward={best_mean_reward:.3f}\n", log_handle)

        # Save the final policy independently from its monitored performance.
        if save_policy:
            _save_named_policy(agent, f"last_policy{policy_suffix}.pth")
            _write_log(f"Last policy saved after episode {episodes}. Best policy: episode {best_policy_episode}, mean learning reward={best_mean_reward:.3f}\n", log_handle)
    finally:
        # Always close the log, including when training raises an exception.
        if log_handle:
            log_handle.close()

    # Return named histories to avoid ambiguous tuple positions.
    return _build_training_results(histories, initial_acceptance_history, buffer_histories, automaton_states, best_mean_reward, best_policy_episode)


# ==============================
# Experiment setup and outputs
# ==============================

def main(args):
    """Configure the experiment, run or load training, and generate diagnostic plots."""
    if args.num_seeds <= 0:
        raise ValueError("num_seeds must be greater than zero")
    # Keep every artifact isolated under results/<experiment-name>/.
    experiment_dir = os.path.join(FRAMEWORK_DIR, "results", args.experiment_name)
    if args.post_process and not os.path.isdir(experiment_dir):
        raise FileNotFoundError(f"Experiment directory not found: {experiment_dir}")
    data_dir = os.path.join(experiment_dir, "results")
    image_dir = os.path.join(experiment_dir, "img")
    log_dir = os.path.join(experiment_dir, "logs")
    policy_dir = os.path.join(experiment_dir, "policy")
    best_policy_dir = os.path.join(policy_dir, "best")
    last_policy_dir = os.path.join(policy_dir, "last")
    for directory in (
        data_dir,
        image_dir,
        log_dir,
        best_policy_dir,
        last_policy_dir,
    ):
        os.makedirs(directory, exist_ok=True)
    _organize_policy_files(policy_dir)
    _organize_legacy_seed_plots(image_dir)
    plot_dir = image_dir
    print(f"Experiment outputs: {experiment_dir}")

    # Load the temporal task and optional training parameters.
    config_path = _resolve_config_path(
        args.config, "trajectory.json", experiment_dir, args.post_process
    )
    abstraction_config_path = _resolve_config_path(
        args.abstraction_config,
        "abstraction.json",
        experiment_dir,
        args.post_process,
    )
    print(f"Task configuration: {config_path}")
    print(f"Abstraction configuration: {abstraction_config_path}")
    with config_path.open(encoding="utf-8") as config_file:
        config = json.load(config_file)
    abstraction_config = AbstractionConfig.load(abstraction_config_path)
    if not args.post_process:
        _archive_config(config_path, experiment_dir, "trajectory.json")
        _archive_config(abstraction_config_path, experiment_dir, "abstraction.json")

    formula = config.get("formula", "F(goal)")
    waypoints = {name: tuple(coordinates) for name, coordinates in config.get("waypoints_dict", {"goal": [5, 0]}).items()}
    gamma = float(config.get("gamma", 0.99))
    goal_reward = float(config.get("goal_reward", 10000))
    primary_level = abstraction_config.primary

    # Build the DFA once for both training and post-processing.
    automaton = LTLfAutomaton(formula)
    validation_report = validate_automaton(
        automaton,
        waypoints,
        width=primary_level.width,
        height=primary_level.height,
    )
    level_summary = ", ".join(
        f"{index}:{level.name}={level.width}x{level.height}"
        for index, level in enumerate(abstraction_config.levels, start=1)
    )
    print(
        "=== LTLf MULTI-EPSILON TRAINING ===\n"
        f"Epsilon strategy: {args.epsilon_strategy}\n"
        f"Network: {args.network_architecture} {args.network_type}\n"
        f"Formula: {formula}\n"
        f"Waypoints: {waypoints}\n"
        f"Abstractions: {level_summary}\n"
        f"Inter-level shaping scale: "
        f"{abstraction_config.inter_level_shaping_scale}\n"
        "Automaton coordinates and training potential: level1\n"
        f"DFA: states={automaton.states}, pre-trace={automaton.initial_state}, "
        f"accepting={sorted(automaton.accepting_states)}\n"
        "Gym reward is ignored by design.\n"
        f"{validation_report.format()}"
    )

    if not args.post_process:
        automaton.render_graph(directory=image_dir)

    # Heatmaps depend only on the saved task configuration, not on agent training.
    multilevel_mdp = MultiLevelWaypointMDP(
        waypoints_dict=waypoints,
        ltlf_automaton=automaton,
        abstraction_config=abstraction_config,
        gamma=gamma,
        goal_reward=goal_reward,
    )
    multilevel_mdp.compute_value_functions()
    save_multilevel_heatmaps(
        multilevel_mdp,
        filename_prefix="multi_epsilon_exp",
        output_root=os.path.join(image_dir, "heatmaps"),
    )
    abstract_mdp = multilevel_mdp.primary_mdp

    if not args.post_process:
        # Create LunarLander only when agent training is requested.
        seeds = [args.seed + index for index in range(args.num_seeds)]
        seed_metrics = []
        for run_index, run_seed in enumerate(seeds, start=1):
            print(f"\n=== SEED RUN {run_index}/{args.num_seeds}: seed={run_seed} ===")
            _set_training_seed(run_seed)
            env = gym.make("LunarLander-v3", continuous=False)
            try:
                _set_training_seed(run_seed, env)
                agent = HierarchicalDQNLearner(
                    env=env,
                    max_episodes=args.episodes,
                    eps_decay=args.eps_decay,
                    gamma=gamma,
                    extra_state_dims=len(automaton.states),
                    use_polyak=args.polyak,
                    tau=args.polyak_tau,
                    target_update_freq=args.target_update_freq,
                    network_architecture=args.network_architecture,
                    network_type=args.network_type,
                    policy_dir=policy_dir,
                )
                policy_suffix = "" if args.num_seeds == 1 else f"_seed_{run_seed}"
                metrics = run_sequential_training(env=env, agent=agent, abstract_mdp=abstract_mdp, episodes=args.episodes, goal_reward=goal_reward, use_shaping=not args.no_shaping, K=args.shaping_scale, log_file=f"{log_dir}/multi_epsilon_training_seed_{run_seed}.log", log_interval=args.log_interval, epsilon_strategy=args.epsilon_strategy, training_shaping_gamma=args.training_shaping_gamma, seed=run_seed, policy_suffix=policy_suffix)
                seed_metrics.append(metrics)
                save_training_data(f"{data_dir}/multi_epsilon_data_seed_{run_seed}.npz", **metrics)
            finally:
                env.close()
        save_training_data(f"{data_dir}/multi_epsilon_data.npz", **_aggregate_seed_metrics(seed_metrics, seeds))

    # Load saved metrics and generate the final diagnostic plots.
    data_path = (
        _resolve_metrics_path(experiment_dir, "multi_epsilon_data.npz")
        if args.post_process
        else Path(data_dir) / "multi_epsilon_data.npz"
    )
    print(f"Training data: {data_path}")
    data = np.load(data_path, allow_pickle=False)
    task_reward_runs = data["task_rewards_runs"] if "task_rewards_runs" in data else data["task_rewards"][np.newaxis, :]
    learning_reward_runs = data["learning_rewards_runs"] if "learning_rewards_runs" in data else data["learning_rewards"][np.newaxis, :]
    epsilon_runs = data["epsilon_history_runs"] if "epsilon_history_runs" in data else data["epsilon_history"][np.newaxis, ...]
    buffer_runs = data["buffer_histories_runs"] if "buffer_histories_runs" in data else data["buffer_histories"][np.newaxis, ...]
    seed_values = data["seeds"] if "seeds" in data else np.asarray([args.seed])
    for obsolete_name in ("buffer_fractions_multi_epsilon.png", "reward_breakdown_multi_epsilon.png"):
        (Path(plot_dir) / obsolete_name).unlink(missing_ok=True)
    for run_index, (run_seed, task_rewards, learning_rewards, epsilon_history) in enumerate(zip(seed_values, task_reward_runs, learning_reward_runs, epsilon_runs)):
        seed_plot_dir = os.path.join(plot_dir, f"seed_{int(run_seed)}")
        os.makedirs(seed_plot_dir, exist_ok=True)
        plot_shaping_reward_breakdown(task_rewards, learning_rewards, epsilon_history, window_size=args.plot_window, filename=f"{seed_plot_dir}/reward_breakdown_multi_epsilon.png", title=f"Reward Breakdown — Seed {int(run_seed)}")
        if run_index < len(buffer_runs):
            plot_buffer_fractions(buffer_runs[run_index], filename=f"{seed_plot_dir}/buffer_fractions_multi_epsilon.png", window_size=args.plot_window, state_labels=data["automaton_states"], title=f"Replay Buffer Composition — Seed {int(run_seed)}")
    plot_training_variance(
        learning_reward_runs,
        window_size=args.plot_window,
        filename=f"{plot_dir}/training_variance_multi_epsilon.png",
        epsilon_histories=epsilon_runs,
        epsilon_labels=data["automaton_states"],
    )
    plot_buffer_variance(buffer_runs, window_size=args.plot_window, filename=f"{plot_dir}/buffer_variance_multi_epsilon.png", state_labels=data["automaton_states"])
    print("\nFinished.")


# ==============================
# Command-line entry point
# ==============================

if __name__ == "__main__":
    # Expose the main training and post-processing options.
    parser = argparse.ArgumentParser(description="Configurable LTLf DDQN training with one epsilon per DFA state.")
    parser.add_argument("--experiment-name", type=_experiment_name, required=True, help="Output directory name under results/.")
    parser.add_argument("--episodes", type=int, default=1000)
    parser.add_argument("--num-seeds", type=_positive_int, default=1, help="Number of training runs with consecutive seeds.")
    parser.add_argument("--seed", type=int, default=42, help="First training seed.")
    parser.add_argument("--config", default="trajectory.json")
    parser.add_argument(
        "--abstraction-config",
        default="abstraction.json",
        help="Ordered grid hierarchy (level1 defines automaton coordinates).",
    )
    parser.add_argument("--eps-decay", type=float, default=0.999)
    parser.add_argument(
        "--epsilon-strategy",
        choices=sorted(EPSILON_STRATEGIES),
        default="cascade",
        help="Use threshold-gated cascade decay or decay each state used during the episode.",
    )
    parser.add_argument("--shaping-scale", type=float, default=1.0)
    parser.add_argument("--log-interval", type=int, default=100)
    parser.add_argument("--plot-window", type=int, default=500)
    parser.add_argument(
        "--polyak",
        action=argparse.BooleanOptionalAction,
        default=True,
        help="Use Polyak target updates (disable with --no-polyak).",
    )
    parser.add_argument("--polyak-tau", type=float, default=0.005)
    parser.add_argument(
        "--target-update-freq",
        type=int,
        default=1000,
        help="Hard target-network update interval used with --no-polyak.",
    )
    parser.add_argument(
        "--network-architecture",
        choices=["classic", "multi-head"],
        default="multi-head",
        help="Use one classic shared output or one output head per DFA state.",
    )
    parser.add_argument(
        "--network-type",
        choices=["standard", "dueling"],
        default="standard",
        help="Use standard Q outputs or dueling value/advantage outputs.",
    )
    parser.add_argument(
        "--training-shaping-gamma",
        action=argparse.BooleanOptionalAction,
        default=True,
        help="Use gamma*Phi(next)-Phi(state) during training; disable to use Phi(next)-Phi(state).",
    )
    parser.add_argument("--no-shaping", action="store_true")
    parser.add_argument("--post-process", action="store_true")
    main(parser.parse_args())


## 9. Write grid visualization and evaluation modules

In [ ]:
%%writefile grid_overlay.py
"""Visualise the abstract grid on top of the LunarLander environment.

The conversion used here is the inverse of ``utils.phi_mapping_grid``.  Grid
cells in the resulting image therefore represent exactly the abstract states
used by the trainer, rather than an evenly spaced, screen-only decoration.
"""

from __future__ import annotations

import argparse
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Mapping, Sequence

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle

from abstraction import AbstractionConfig, map_waypoints
from utils import phi_mapping_grid, spatial_grid_boundaries


SCRIPT_DIR = Path(__file__).resolve().parent
FRAMEWORK_DIR = SCRIPT_DIR.parent
DEFAULT_CONFIG = SCRIPT_DIR / "trajectory.json"
DEFAULT_ABSTRACTION_CONFIG = SCRIPT_DIR / "abstraction.json"


@dataclass(frozen=True)
class LunarLanderGeometry:
    """Screen/world constants required to project observations onto a frame."""

    viewport_width: int
    viewport_height: int
    scale: float
    helipad_y: float
    leg_down: float


def geometry_from_env(env: gym.Env) -> LunarLanderGeometry:
    """Read projection constants from a reset LunarLander environment."""
    from gymnasium.envs.box2d import lunar_lander

    base_env = env.unwrapped
    if not hasattr(base_env, "helipad_y"):
        raise TypeError("The supplied environment is not a LunarLander environment")
    return LunarLanderGeometry(
        viewport_width=lunar_lander.VIEWPORT_W,
        viewport_height=lunar_lander.VIEWPORT_H,
        scale=lunar_lander.SCALE,
        helipad_y=float(base_env.helipad_y),
        leg_down=lunar_lander.LEG_DOWN,
    )


def observation_to_pixel(
    observation: Sequence[float], geometry: LunarLanderGeometry
) -> tuple[float, float]:
    """Project LunarLander's normalised (x, y) observation onto RGB pixels."""
    half_world_width = geometry.viewport_width / geometry.scale / 2.0
    half_world_height = geometry.viewport_height / geometry.scale / 2.0
    world_x = (float(observation[0]) + 1.0) * half_world_width
    world_y = (
        float(observation[1]) * half_world_height
        + geometry.helipad_y
        + geometry.leg_down / geometry.scale
    )
    pixel_x = world_x * geometry.scale
    pixel_y = geometry.viewport_height - world_y * geometry.scale
    return pixel_x, pixel_y


def pixel_to_observation(
    pixel_x: float,
    pixel_y: float,
    geometry: LunarLanderGeometry,
) -> tuple[float, float]:
    """Invert the frame projection for the two discretised coordinates."""
    half_world_width = geometry.viewport_width / geometry.scale / 2.0
    half_world_height = geometry.viewport_height / geometry.scale / 2.0
    world_x = float(pixel_x) / geometry.scale
    world_y = (geometry.viewport_height - float(pixel_y)) / geometry.scale
    observation_x = world_x / half_world_width - 1.0
    observation_y = (
        world_y - geometry.helipad_y - geometry.leg_down / geometry.scale
    ) / half_world_height
    return observation_x, observation_y


def _grid_boundaries(
    grid_w: int,
    grid_h: int,
    geometry: LunarLanderGeometry,
) -> tuple[np.ndarray, np.ndarray]:
    """Return the boundaries implied by the active spatial discretizer."""
    x_normalised, y_normalised = spatial_grid_boundaries(grid_w, grid_h)
    x_pixels = np.array(
        [observation_to_pixel((x, 0.0), geometry)[0] for x in x_normalised]
    )
    y_pixels = np.array(
        [observation_to_pixel((0.0, y), geometry)[1] for y in y_normalised]
    )

    # phi_mapping_grid clips observations outside its nominal domain into its
    # edge cells. Extend those cells to the RGB viewport edges whenever the
    # corresponding border observation maps to index 0 or to the last index.
    # Internal boundaries remain entirely inferred from the active mapper.
    left_observation_x, top_observation_y = pixel_to_observation(
        0.0, 0.0, geometry
    )
    right_observation_x, bottom_observation_y = pixel_to_observation(
        geometry.viewport_width, geometry.viewport_height, geometry
    )
    if phi_mapping_grid((left_observation_x, 0.0), grid_w, grid_h)[0] == 0:
        x_pixels[0] = min(x_pixels[0], 0.0)
    if phi_mapping_grid((right_observation_x, 0.0), grid_w, grid_h)[0] == grid_w - 1:
        x_pixels[-1] = max(x_pixels[-1], float(geometry.viewport_width))
    if phi_mapping_grid((0.0, bottom_observation_y), grid_w, grid_h)[1] == 0:
        y_pixels[0] = max(y_pixels[0], float(geometry.viewport_height))
    if phi_mapping_grid((0.0, top_observation_y), grid_w, grid_h)[1] == grid_h - 1:
        y_pixels[-1] = min(y_pixels[-1], 0.0)

    return x_pixels, y_pixels


def abstract_cell_to_pixel(
    grid_x: int,
    grid_y: int,
    grid_w: int,
    grid_h: int,
    geometry: LunarLanderGeometry,
) -> tuple[float, float]:
    """Return the pixel coordinates of an abstract cell's centre."""
    if not (0 <= grid_x < grid_w and 0 <= grid_y < grid_h):
        raise ValueError(f"Abstract cell ({grid_x}, {grid_y}) is outside the grid")
    x_lines, y_lines = _grid_boundaries(grid_w, grid_h, geometry)
    return (
        float((x_lines[grid_x] + x_lines[grid_x + 1]) / 2.0),
        float((y_lines[grid_y] + y_lines[grid_y + 1]) / 2.0),
    )


def draw_abstract_grid(
    frame: np.ndarray,
    geometry: LunarLanderGeometry,
    grid_w: int,
    grid_h: int,
    waypoints: Mapping[str, Sequence[int]] | None = None,
    observation: Sequence[float] | None = None,
    title: str = "LunarLander with Abstract Grid",
):
    """Create a figure containing the effective clipped grid and its markers."""
    if grid_w < 2 or grid_h < 2:
        raise ValueError("grid_w and grid_h must both be at least 2")

    figure, axis = plt.subplots(figsize=(7.2, 4.8), constrained_layout=True)
    axis.imshow(frame)
    x_lines, y_lines = _grid_boundaries(grid_w, grid_h, geometry)
    x_centres = (x_lines[:-1] + x_lines[1:]) / 2.0
    y_centres = (y_lines[:-1] + y_lines[1:]) / 2.0
    grid_color = "#ff1744"

    for x_pixel in x_lines:
        axis.axvline(x_pixel, color=grid_color, linewidth=1.6, alpha=0.95)
    for y_pixel in y_lines:
        axis.axhline(y_pixel, color=grid_color, linewidth=1.6, alpha=0.95)

    # The mapping clips everything outside its stated observation domain into
    # an edge cell; tint the currently occupied abstract cell when requested.
    if observation is not None:
        abstract_x, abstract_y = phi_mapping_grid(observation, grid_w, grid_h)
        x0, x1 = sorted((x_lines[abstract_x], x_lines[abstract_x + 1]))
        y0, y1 = sorted((y_lines[abstract_y], y_lines[abstract_y + 1]))
        x0, x1 = np.clip((x0, x1), 0, geometry.viewport_width)
        y0, y1 = np.clip((y0, y1), 0, geometry.viewport_height)
        axis.add_patch(
            Rectangle(
                (x0, y0),
                x1 - x0,
                y1 - y0,
                facecolor="#00e5ff",
                edgecolor="#00e5ff",
                linewidth=2.5,
                alpha=0.25,
                label=f"Current cell ({abstract_x}, {abstract_y})",
            )
        )

    for name, coordinates in (waypoints or {}).items():
        if len(coordinates) != 2:
            raise ValueError(f"Waypoint {name!r} must contain [x, y]")
        grid_x, grid_y = int(coordinates[0]), int(coordinates[1])
        if not (0 <= grid_x < grid_w and 0 <= grid_y < grid_h):
            raise ValueError(f"Waypoint {name!r} is outside the abstract grid")
        pixel_x, pixel_y = abstract_cell_to_pixel(
            grid_x, grid_y, grid_w, grid_h, geometry
        )
        axis.scatter(pixel_x, pixel_y, s=150, marker="o", color="#ffca28",
                     edgecolor="black", linewidth=1.3, zorder=5)
        axis.annotate(
            f"{name} ({grid_x}, {grid_y})",
            (pixel_x, pixel_y),
            xytext=(7, -10),
            textcoords="offset points",
            color="black",
            fontsize=9,
            fontweight="bold",
            bbox={"boxstyle": "round,pad=0.25", "fc": "#ffca28", "alpha": 0.9},
            zorder=6,
        )

    axis.set_xlim(0, geometry.viewport_width)
    # A small part of the configured y-domain can lie above the RGB viewport.
    # Keep it in view so that no abstract row or coordinate label disappears.
    visible_top = min(0.0, float(np.min(y_lines)))
    axis.set_ylim(geometry.viewport_height, visible_top)
    # Put the abstract coordinates at cell centres. Since image coordinates
    # grow downwards while abstract y grows upwards, y_centres is descending:
    # label 0 consequently appears at the bottom and grid_h - 1 at the top.
    axis.set_xticks(x_centres, labels=range(grid_w))
    axis.set_yticks(y_centres, labels=range(grid_h))
    axis.set_xlabel("Abstract x-coordinate")
    axis.set_ylabel("Abstract y-coordinate")
    axis.tick_params(
        axis="both",
        which="major",
        color=grid_color,
        labelcolor=grid_color,
        labelsize=10,
        width=1.5,
        length=5,
    )
    for label in (*axis.get_xticklabels(), *axis.get_yticklabels()):
        label.set_fontweight("bold")

    if observation is not None:
        axis.legend(
            loc="upper left",
            bbox_to_anchor=(1.02, 1.0),
            borderaxespad=0.0,
            frameon=True,
        )
        figure.tight_layout(rect=(0.0, 0.0, 0.82, 1.0))
    else:
        figure.tight_layout()
    return figure


def generate_overlay(
    output_path: str | Path,
    config_path: str | Path = DEFAULT_CONFIG,
    seed: int | None = 0,
    abstraction_config_path: str | Path = DEFAULT_ABSTRACTION_CONFIG,
    level: int = 1,
) -> Path:
    """Reset LunarLander and save one annotated RGB frame as a PNG."""
    config_path = Path(config_path)
    with config_path.open(encoding="utf-8") as config_file:
        config = json.load(config_file)
    abstraction_config = AbstractionConfig.load(abstraction_config_path)
    if not 1 <= level <= len(abstraction_config.levels):
        raise ValueError(
            f"level must be between 1 and {len(abstraction_config.levels)}"
        )
    selected_level = abstraction_config.levels[level - 1]
    primary_level = abstraction_config.primary
    primary_waypoints = {
        name: tuple(coordinates)
        for name, coordinates in config.get("waypoints_dict", {}).items()
    }
    waypoints = (
        primary_waypoints
        if level == 1
        else map_waypoints(
            primary_waypoints,
            primary_level.width,
            primary_level.height,
            selected_level.width,
            selected_level.height,
        )
    )

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    env = gym.make("LunarLander-v3", continuous=False, render_mode="rgb_array")
    try:
        observation, _ = env.reset(seed=seed)
        frame = env.render()
        geometry = geometry_from_env(env)
        figure = draw_abstract_grid(
            frame=frame,
            geometry=geometry,
            grid_w=selected_level.width,
            grid_h=selected_level.height,
            waypoints=waypoints,
            observation=observation,
            title=(
                f"LunarLander — level{level} ({selected_level.name}, "
                f"{selected_level.width}x{selected_level.height})"
            ),
        )
        figure.savefig(output_path, dpi=300, bbox_inches="tight")
        plt.close(figure)
    finally:
        env.close()
    return output_path.resolve()


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Generate a LunarLander frame with the abstract grid overlaid."
    )
    parser.add_argument("--config", type=Path, default=DEFAULT_CONFIG)
    parser.add_argument(
        "--abstraction-config",
        type=Path,
        default=DEFAULT_ABSTRACTION_CONFIG,
    )
    parser.add_argument(
        "--level",
        type=int,
        default=1,
        help="1-based abstraction level to draw (default: 1).",
    )
    parser.add_argument("--output", type=Path)
    parser.add_argument("--seed", type=int, default=0)
    args = parser.parse_args()
    output_path = args.output or FRAMEWORK_DIR / "results" / "abstract_grid_overlay.png"
    saved_path = generate_overlay(
        output_path,
        args.config,
        args.seed,
        args.abstraction_config,
        args.level,
    )
    print(f"Image saved to: {saved_path}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile evaluate.py
"""Evaluate classic or multi-head LTLf-guided LunarLander DQN policies."""

# ==============================
# Standard library imports
# ==============================

import argparse
import json
import time
from pathlib import Path

# ==============================
# External and project imports
# ==============================

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch

from abstraction import AbstractionConfig
from abstract_mdps import LTLfAutomaton, LTLfWaypointMDP
from agent import (
    DuelingQNetwork,
    MultiHeadDuelingQNetwork,
    MultiHeadQNetwork,
    QNetwork,
)
from grid_overlay import (
    abstract_cell_to_pixel,
    draw_abstract_grid,
    geometry_from_env,
)
from utils import (
    LEARNING_REWARD_COLOR,
    RAW_DATA_COLOR,
    SERIES_COLORS,
    phi_mapping_sequential,
)


# ==============================
# Paths and generic helpers
# ==============================

SCRIPT_DIR = Path(__file__).resolve().parent
FRAMEWORK_DIR = SCRIPT_DIR.parent
EXPERIMENTS_DIR = FRAMEWORK_DIR / "results"


def moving_average(data, window_size):
    """Return a moving average, or the original values when the window is larger."""
    values = np.asarray(data, dtype=np.float64)
    if len(values) < window_size:
        return values
    return np.convolve(values, np.ones(window_size) / window_size, mode="valid")


def _resolve_policy_path(policy, policy_dir):
    """Accept explicit paths as well as filenames relative to the policy directory."""
    supplied_path = Path(policy).expanduser()
    if supplied_path.is_file():
        return supplied_path.resolve()

    policy_root = Path(policy_dir).expanduser()
    for policy_path in (
        policy_root / supplied_path,
        policy_root / "best" / supplied_path,
        policy_root / "last" / supplied_path,
    ):
        if policy_path.is_file():
            return policy_path.resolve()

    raise FileNotFoundError(f"Policy '{policy}' not found either as an explicit path or under '{policy_dir}'.")


def _load_state_dict(policy_path, device):
    """Load both plain state dictionaries and common wrapped checkpoints."""
    checkpoint = torch.load(policy_path, map_location=device, weights_only=True)
    if isinstance(checkpoint, dict):
        for key in ("policy_state_dict", "state_dict", "model_state_dict"):
            if key in checkpoint:
                return checkpoint[key]
    return checkpoint


def _abstract_position(observation, q, grid_w, grid_h):
    """Map an environment observation to its abstract grid coordinates."""
    x, y, _ = phi_mapping_sequential(observation, q, grid_w, grid_h)
    return x, y


# ==============================
# Policy evaluation
# ==============================

def evaluate_policy(policy, policy_dir, episodes, render, formula, waypoints_dict,
                    goal_reward, grid_w, grid_h, seed, trace_episodes=0,
                    network_architecture="multi-head", network_type="standard"):
    """Load and evaluate one policy using the same DFA semantics as training."""
    # Rebuild the same automaton and abstract MDP used during training.
    policy_path = _resolve_policy_path(policy, policy_dir)
    policy_name = policy_path.name
    automaton = LTLfAutomaton(formula)
    abstract_mdp = LTLfWaypointMDP(waypoints_dict=waypoints_dict, ltlf_automaton=automaton, width=grid_w, height=grid_h, goal_reward=goal_reward)
    automaton_states = list(automaton.states)
    state_to_index = {q: index for index, q in enumerate(automaton_states)}

    # Create the same classic or multi-head network used during training.
    render_mode = "human" if render else ("rgb_array" if trace_episodes else None)
    env = gym.make("LunarLander-v3", continuous=False, render_mode=render_mode)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if network_type not in {"standard", "dueling"}:
        raise ValueError("network_type must be one of: standard, dueling")
    if network_architecture not in {"classic", "multi-head"}:
        raise ValueError("network_architecture must be one of: classic, multi-head")
    if network_architecture == "multi-head":
        network_cls = MultiHeadDuelingQNetwork if network_type == "dueling" else MultiHeadQNetwork
        network = network_cls(
            env.observation_space.shape[0],
            env.action_space.n,
            len(automaton_states),
        ).to(device)
    else:
        network_cls = DuelingQNetwork if network_type == "dueling" else QNetwork
        network = network_cls(
            env.observation_space.shape[0] + len(automaton_states),
            env.action_space.n,
        ).to(device)

    # Load the trained parameters before starting any episode.
    try:
        network.load_state_dict(_load_state_dict(policy_path, device))
        network.eval()
    except Exception:
        env.close()
        raise

    task_returns = []
    environment_returns = []
    episode_lengths = []
    successes = 0
    state_reach_counts = {q: 0 for q in automaton_states}
    grid_traces = []
    trace_frames = []
    trace_geometries = []

    # Run every requested episode sequentially.
    try:
        for episode in range(episodes):
            episode_seed = None if seed is None else seed + episode
            observation, _ = env.reset(seed=episode_seed)
            tracing = episode < trace_episodes
            if tracing:
                trace_frames.append(env.render())
                trace_geometries.append(geometry_from_env(env))
                initial_cell = _abstract_position(observation, automaton.get_initial_q(), grid_w, grid_h)
                cell_trace = [initial_cell]

            # Training consumes the valuation at s0 before choosing the first action.
            initial_q = automaton.get_initial_q()
            initial_x, initial_y = _abstract_position(observation, initial_q, grid_w, grid_h)
            initial_truth_assignment = abstract_mdp._get_truth_assignment(initial_x, initial_y)
            q = automaton.get_next_q(initial_q, initial_truth_assignment)
            if q not in state_to_index:
                raise RuntimeError(f"DFA returned unknown state {q!r}")

            reached_states = {q}
            success = automaton.is_goal_reached(q)
            terminated = truncated = False
            environment_return = 0.0
            steps = 0

            while not (success or terminated or truncated):
                # Evaluation is greedy. The classic architecture consumes the
                # augmented state; multi-head routes the physical state by q.
                with torch.inference_mode():
                    if network_architecture == "multi-head":
                        state_tensor = torch.as_tensor(
                            observation,
                            dtype=torch.float32,
                            device=device,
                        ).unsqueeze(0)
                        head_index = torch.tensor(
                            [state_to_index[q]],
                            dtype=torch.long,
                            device=device,
                        )
                        q_values = network(state_tensor, head_index)
                    else:
                        one_hot = np.zeros(
                            len(automaton_states),
                            dtype=np.float32,
                        )
                        one_hot[state_to_index[q]] = 1.0
                        augmented_state = np.concatenate(
                            (observation, one_hot)
                        ).astype(np.float32)
                        state_tensor = torch.as_tensor(
                            augmented_state,
                            dtype=torch.float32,
                            device=device,
                        ).unsqueeze(0)
                        q_values = network(state_tensor)
                    action = q_values.argmax(dim=1).item()

                next_observation, env_reward, terminated, truncated, _ = env.step(action)
                environment_return += float(env_reward)
                steps += 1

                # Advance the DFA using the propositions true in the arrival state.
                x, y = _abstract_position(next_observation, q, grid_w, grid_h)
                if tracing and (x, y) != cell_trace[-1]:
                    cell_trace.append((x, y))
                truth_assignment = abstract_mdp._get_truth_assignment(x, y)
                next_q = automaton.get_next_q(q, truth_assignment)
                if next_q not in state_to_index:
                    raise RuntimeError(f"DFA returned unknown state {next_q!r}")

                # Report every effective DFA transition during evaluation.
                if next_q != q:
                    if automaton.is_goal_reached(next_q):
                        print(f"[{policy_name} | Episode {episode + 1}] DFA transition {q} -> {next_q}: final goal reached.")
                    else:
                        print(f"[{policy_name} | Episode {episode + 1}] DFA transition {q} -> {next_q}: intermediate waypoint reached.")

                reached_states.add(next_q)

                observation = next_observation
                q = next_q
                success = automaton.is_goal_reached(q)

                if render:
                    time.sleep(0.02)

            # Store episode-level metrics and count every DFA state reached at least once.
            successes += int(success)
            for reached_q in reached_states:
                state_reach_counts[reached_q] += 1
            task_returns.append(float(goal_reward) if success else 0.0)
            environment_returns.append(environment_return)
            episode_lengths.append(steps)
            if tracing:
                grid_traces.append(cell_trace)
    finally:
        env.close()

    return {
        "policy": policy_name,
        "path": str(policy_path),
        "task_returns": task_returns,
        "environment_returns": environment_returns,
        "episode_lengths": episode_lengths,
        "successes": successes,
        "state_reach_counts": state_reach_counts,
        "grid_traces": grid_traces,
        "trace_frames": trace_frames,
        "trace_geometries": trace_geometries,
    }


# ==============================
# Plotting helpers
# ==============================

def _safe_stem(name):
    """Create a filesystem-safe plot stem from a checkpoint filename."""
    return "".join(character if character.isalnum() or character in "-_." else "_" for character in Path(name).stem)


def plot_policy(result, window_size, output_dir):
    """Plot Gym returns for one policy."""
    returns = result["environment_returns"]
    smooth = moving_average(returns, window_size)

    figure, axis = plt.subplots(figsize=(7.2, 4.4), constrained_layout=True)
    episodes = np.arange(1, len(returns) + 1)
    axis.plot(episodes, returns, alpha=0.28, color=RAW_DATA_COLOR, linewidth=0.8, label="Raw Gym return")
    start = window_size - 1 if len(returns) >= window_size else 0
    axis.plot(np.arange(start + 1, start + len(smooth) + 1), smooth, color=LEARNING_REWARD_COLOR, linewidth=1.7, label=f"Trailing mean (N={window_size})")
    axis.set_xlabel("#Episode")
    axis.set_ylabel("Gym return")
    axis.spines["top"].set_visible(True)
    axis.spines["right"].set_visible(True)
    axis.grid(axis="y", color="#d9d9d9", linewidth=0.6, alpha=0.8)
    axis.legend(loc="lower center", bbox_to_anchor=(0.5, 1.01), ncol=2, frameon=False)
    output_path = output_dir / f"eval_{_safe_stem(result['policy'])}.png"
    figure.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close(figure)
    return output_path


def plot_comparison(results, window_size, output_dir):
    """Plot smoothed Gym returns for multiple policies."""
    figure, axis = plt.subplots(figsize=(7.2, 4.4), constrained_layout=True)
    for index, result in enumerate(results):
        returns = result["environment_returns"]
        smooth = moving_average(returns, window_size)
        start = window_size - 1 if len(returns) >= window_size else 0
        axis.plot(np.arange(start + 1, start + len(smooth) + 1), smooth, color=SERIES_COLORS[index % len(SERIES_COLORS)], linewidth=1.7, label=result["policy"])
    axis.set_xlabel("#Episode")
    axis.set_ylabel("Gym return")
    axis.spines["top"].set_visible(True)
    axis.spines["right"].set_visible(True)
    axis.grid(axis="y", color="#d9d9d9", linewidth=0.6, alpha=0.8)
    axis.legend(loc="lower center", bbox_to_anchor=(0.5, 1.01), ncol=min(len(results), 3), frameon=False)
    output_path = output_dir / "policy_comparison.png"
    figure.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close(figure)
    return output_path


def plot_grid_traces(result, waypoints_dict, grid_w, grid_h, output_dir):
    """Save one abstract-grid path image for every recorded episode."""
    output_paths = []
    trace_data = zip(
        result["grid_traces"],
        result["trace_frames"],
        result["trace_geometries"],
    )
    for episode_index, (cells, frame, geometry) in enumerate(trace_data, start=1):
        figure = draw_abstract_grid(
            frame=frame,
            geometry=geometry,
            grid_w=grid_w,
            grid_h=grid_h,
            waypoints=waypoints_dict,
            title=f"Agent Abstract-Cell Trace — Episode {episode_index}",
        )
        axis = figure.axes[0]
        points = [
            abstract_cell_to_pixel(x, y, grid_w, grid_h, geometry)
            for x, y in cells
        ]
        if points:
            pixel_x, pixel_y = zip(*points)
            axis.plot(
                pixel_x,
                pixel_y,
                color="#00e5ff",
                linewidth=2.8,
                marker="o",
                markersize=5,
                label="Visited-cell path",
                zorder=4,
            )
            for change_index, ((cell_x, cell_y), (point_x, point_y)) in enumerate(
                zip(cells, points)
            ):
                axis.annotate(
                    str(change_index),
                    (point_x, point_y),
                    ha="center",
                    va="center",
                    fontsize=7,
                    fontweight="bold",
                    color="black",
                    zorder=7,
                )
        axis.legend(
            loc="upper left",
            bbox_to_anchor=(1.02, 1.0),
            borderaxespad=0.0,
            frameon=True,
        )
        figure.tight_layout(rect=(0.0, 0.0, 0.82, 1.0))
        output_path = output_dir / (
            f"grid_trace_{_safe_stem(result['policy'])}_episode_{episode_index}.png"
        )
        figure.savefig(output_path, dpi=300, bbox_inches="tight")
        plt.close(figure)
        output_paths.append(output_path)
    return output_paths


def format_waypoint_trace(cells, waypoints_dict):
    """Report the first cell-change index at which each waypoint was visited."""
    first_visit = {}
    for index, cell in enumerate(cells):
        first_visit.setdefault(tuple(cell), index)
    return ", ".join(
        f"{name}=reached@{first_visit[tuple(position)]}"
        if tuple(position) in first_visit
        else f"{name}=missed"
        for name, position in waypoints_dict.items()
    )


# ==============================
# Command-line interface
# ==============================

def _positive_int(value):
    """Parse and validate a strictly positive integer."""
    parsed = int(value)
    if parsed <= 0:
        raise argparse.ArgumentTypeError("must be greater than zero")
    return parsed


def _select_files_graphically(policy_dir, config_path):
    """Select policy checkpoints and the experiment configuration with native dialogs."""
    try:
        import tkinter as tk
        from tkinter import filedialog
    except ImportError as error:
        raise RuntimeError(
            "The graphical selector requires tkinter. Install python3-tk or pass "
            "the policy paths and --config from the command line."
        ) from error

    try:
        root = tk.Tk()
    except tk.TclError as error:
        raise RuntimeError(
            "The graphical selector could not be opened. Make sure a desktop "
            "session is available, or use the command-line arguments."
        ) from error
    root.withdraw()
    root.update()

    try:
        initial_directory = EXPERIMENTS_DIR if EXPERIMENTS_DIR.is_dir() else SCRIPT_DIR
        policies = filedialog.askopenfilenames(
            parent=root,
            title="Select one or more policy files",
            initialdir=str(initial_directory),
            filetypes=[
                ("PyTorch checkpoints", "*.pt *.pth *.ckpt"),
                ("All files", "*"),
            ],
        )
        if not policies:
            raise RuntimeError("No policy file was selected.")

        config = filedialog.askopenfilename(
            parent=root,
            title="Select trajectory.json",
            initialdir=str(initial_directory),
            initialfile=Path(config_path).name,
            filetypes=[
                ("JSON files", "*.json"),
                ("All files", "*"),
            ],
        )
        if not config:
            raise RuntimeError("No trajectory configuration was selected.")
    finally:
        root.destroy()

    return list(policies), Path(config)


def parse_args():
    """Build and parse the evaluator command-line arguments."""
    parser = argparse.ArgumentParser(description="Evaluate classic or multi-head LTLf-guided DQN policies for LunarLander.")
    parser.add_argument(
        "policies",
        nargs="*",
        help="Checkpoint filenames or explicit checkpoint paths. If omitted, graphical file selectors are opened.",
    )
    parser.add_argument("--config", type=Path, default=SCRIPT_DIR / "trajectory.json", help="Experiment JSON configuration.")
    parser.add_argument(
        "--abstraction-config",
        type=Path,
        default=SCRIPT_DIR / "abstraction.json",
        help="Grid hierarchy; evaluation uses its level1 dimensions.",
    )
    parser.add_argument("--policy-dir", type=Path, default=FRAMEWORK_DIR / "results", help="Directory used to resolve checkpoint filenames.")
    parser.add_argument("--gui", action="store_true", help="Select policies and trajectory.json using graphical dialogs.")
    parser.add_argument("--episodes", type=_positive_int, default=100)
    parser.add_argument("--window", type=_positive_int, default=10)
    parser.add_argument("--seed", type=int, default=None)
    parser.add_argument("--render", action="store_true")
    parser.add_argument(
        "--trace-grid",
        action="store_true",
        help="Save the sequence of abstract cells visited during evaluation.",
    )
    parser.add_argument(
        "--trace-episodes",
        type=_positive_int,
        default=1,
        help="Number of episodes to trace when --trace-grid is enabled (default: 1).",
    )
    parser.add_argument("--output-dir", type=Path, default=FRAMEWORK_DIR / "results" / "evaluation")
    parser.add_argument(
        "--network-architecture",
        choices=["classic", "multi-head"],
        default="multi-head",
        help="Network architecture used by the checkpoint.",
    )
    parser.add_argument(
        "--network-type",
        choices=["standard", "dueling"],
        default="standard",
        help="Standard or dueling network type used by the checkpoint.",
    )
    return parser.parse_args()


# ==============================
# Main program
# ==============================

def main():
    """Load the configuration, evaluate the policies, and generate the plots."""
    args = parse_args()
    if args.render and args.trace_grid:
        raise SystemExit(
            "--render and --trace-grid cannot be used together because Gymnasium "
            "requires a single render mode. Run them as separate evaluations."
        )

    # Open native file dialogs when requested or when no policy was supplied.
    if args.gui or not args.policies:
        try:
            args.policies, args.config = _select_files_graphically(args.policy_dir, args.config)
        except RuntimeError as error:
            raise SystemExit(f"Selection cancelled: {error}") from error

    # Load the LTLf task shared with the trainer.
    with args.config.expanduser().open(encoding="utf-8") as config_file:
        config = json.load(config_file)
    abstraction_config = AbstractionConfig.load(args.abstraction_config.expanduser())

    formula = config["formula"]
    raw_waypoints = config["waypoints_dict"]
    waypoints_dict = {name: tuple(coordinates) for name, coordinates in raw_waypoints.items()}
    grid_w = abstraction_config.primary.width
    grid_h = abstraction_config.primary.height
    goal_reward = float(config.get("goal_reward", 10000.0))

    # Evaluate policies one at a time to keep rendering and output deterministic.
    results = []
    for policy in args.policies:
        traced_episodes = min(args.trace_episodes, args.episodes) if args.trace_grid else 0
        result = evaluate_policy(
            policy, args.policy_dir, args.episodes, args.render, formula,
            waypoints_dict, goal_reward, grid_w, grid_h, args.seed,
            trace_episodes=traced_episodes,
            network_architecture=args.network_architecture,
            network_type=args.network_type,
        )
        results.append(result)

    # Print the summary and create one plot for each evaluated policy.
    args.output_dir.mkdir(parents=True, exist_ok=True)
    for result in results:
        success_rate = result["successes"] / args.episodes
        mean_gym_return = np.mean(result["environment_returns"])
        mean_length = np.mean(result["episode_lengths"])
        reached = ", ".join(f"q={q}: {count}/{args.episodes}" for q, count in result["state_reach_counts"].items())
        print(f"[{result['policy']}] success={success_rate:.1%}, mean Gym return={mean_gym_return:.2f}, mean length={mean_length:.1f} | reached: {reached}")
        print(f"Plot saved to: {plot_policy(result, args.window, args.output_dir)}")
        if args.trace_grid:
            trace_paths = plot_grid_traces(
                result, waypoints_dict, grid_w, grid_h, args.output_dir
            )
            for episode_index, (cells, trace_path) in enumerate(
                zip(result["grid_traces"], trace_paths), start=1
            ):
                waypoint_status = format_waypoint_trace(cells, waypoints_dict)
                print(
                    f"Grid trace episode {episode_index}: {waypoint_status} | "
                    f"saved to: {trace_path}"
                )

    # Add a combined comparison when more than one policy was requested.
    if len(results) > 1:
        print(f"Comparison saved to: {plot_comparison(results, args.window, args.output_dir)}")


if __name__ == "__main__":
    main()


## 10. Configure the temporal task

Waypoint e goal sono sempre espressi nelle coordinate di `level1`.

In [ ]:
%%writefile trajectory.json
{
    "formula": "F(wp1 & X(F(g1)))",
    "goal_reward": 10000,
    "waypoints_dict": {
        "wp1": [1, 8],
        "g1": [8, 8]
    }
}


## 11. Configure the abstraction hierarchy

Modificare liberamente la lista `levels`. Il primo elemento rimane la griglia
usata dall'automa e dal training.

In [ ]:
%%writefile abstraction.json
{
    "inter_level_shaping_scale": 1.0,
    "levels": [
        {
            "name": "level1",
            "grid_w": 12,
            "grid_h": 12
        },
        {
            "name": "level2",
            "grid_w": 6,
            "grid_h": 6
        }
    ]
}


## 12. Configure the training run

Le scelte sono indipendenti:

- `EPSILON_STRATEGY="cascade"`: decadimento con soglia e sblocco progressivo;
- `EPSILON_STRATEGY="visited"`: decadimento una volta per episodio per ogni stato DFA che ha selezionato almeno un’azione;
- `NETWORK_ARCHITECTURE="classic"`: rete condivisa con un’unica uscita;
- `NETWORK_ARCHITECTURE="multi-head"`: encoder fisico condiviso e una testa per stato DFA.

`NETWORK_TYPE` permette inoltre di usare uscite standard oppure dueling in entrambe le architetture.


In [ ]:
EPISODES = 1000
NUM_SEEDS = 1
SEED = 42
EPSILON_DECAY = 0.999
EPSILON_STRATEGY = "cascade"  # "cascade" or "visited"
SHAPING_SCALE = 1.0
TRAINING_USE_GAMMA = True
LOG_INTERVAL = 100
PLOT_WINDOW = 500
DISABLE_SHAPING = False

USE_POLYAK = True
POLYAK_TAU = 0.005
TARGET_UPDATE_FREQ = 1000
NETWORK_ARCHITECTURE = "multi-head"  # "classic" or "multi-head"
NETWORK_TYPE = "standard"  # "standard" or "dueling"

print(f"Episodes: {EPISODES}")
print(f"Number of seeds: {NUM_SEEDS}")
print(f"First seed: {SEED}")
print(f"Epsilon decay: {EPSILON_DECAY}")
print(f"Epsilon strategy: {EPSILON_STRATEGY}")
print(f"Shaping scale: {SHAPING_SCALE}")
print(f"Training uses gamma: {TRAINING_USE_GAMMA}")
print(f"Polyak update: {USE_POLYAK}")
print(f"Network: {NETWORK_ARCHITECTURE} {NETWORK_TYPE}")


## 13. Run training

In [ ]:
import subprocess
import sys

command = [
    sys.executable,
    "-u",
    "trainer.py",
    "--episodes", str(EPISODES),
    "--config", "trajectory.json",
    "--abstraction-config", "abstraction.json",
    "--eps-decay", str(EPSILON_DECAY),
    "--epsilon-strategy", EPSILON_STRATEGY,
    "--shaping-scale", str(SHAPING_SCALE),
    "--log-interval", str(LOG_INTERVAL),
    "--plot-window", str(PLOT_WINDOW),
    "--polyak-tau", str(POLYAK_TAU),
    "--target-update-freq", str(TARGET_UPDATE_FREQ),
    "--network-architecture", NETWORK_ARCHITECTURE,
    "--network-type", NETWORK_TYPE,
]
command.extend(["--num-seeds", str(NUM_SEEDS), "--seed", str(SEED)])
if not TRAINING_USE_GAMMA:
    command.append("--no-training-shaping-gamma")
if DISABLE_SHAPING:
    command.append("--no-shaping")
if not USE_POLYAK:
    command.append("--no-polyak")

environment = os.environ.copy()
environment["MPLBACKEND"] = "Agg"
environment["PYTHONUNBUFFERED"] = "1"
environment["PYTHONHASHSEED"] = str(SEED)

print("Running:", " ".join(command))
process = subprocess.Popen(
    command,
    cwd=WORK_DIR,
    env=environment,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="", flush=True)
return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, command)


## 14. Inspect saved metrics, logs, heatmaps, and policies

In [ ]:
import numpy as np

data_path = WORK_DIR / "results" / "multi_epsilon_data.npz"
metrics = np.load(data_path, allow_pickle=False)

print("Saved metrics:")
for key in metrics.files:
    value = metrics[key]
    print(f"- {key}: shape={value.shape}, dtype={value.dtype}")

print(f"\nEpsilon schedules: {metrics['epsilon_history'].shape[0]}")
print(f"Best policy episode: {int(metrics['best_policy_episode'])}")
print(
    "Best mean learning reward: "
    f"{float(metrics['best_mean_learning_reward']):.3f}"
)
print(f"Overall success rate: {metrics['successes'].mean():.2%}")

log_path = WORK_DIR / "logs" / "multi_epsilon_training.log"
if log_path.exists():
    print("\nLast log lines:\n")
    print("\n".join(log_path.read_text(encoding="utf-8").splitlines()[-30:]))

print("\nGenerated heatmaps:")
for heatmap_path in sorted((WORK_DIR / "img" / "heatmaps").rglob("*.png")):
    print(f"- {heatmap_path.relative_to(WORK_DIR)}")

print("\nSaved policies:")
for policy_path in sorted((WORK_DIR / "policy").glob("*.pth")):
    print(f"- {policy_path.name}")


## 15. Display generated plots

In [ ]:

from IPython.display import display
from PIL import Image

plot_paths = sorted((WORK_DIR / "img").glob("*.png"))
heatmap_paths = sorted((WORK_DIR / "img" / "heatmaps").rglob("*.png"))
for image_path in plot_paths + heatmap_paths:
    print(image_path.relative_to(WORK_DIR))
    display(Image.open(image_path))

## 16. Package outputs for download

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

archive_path = Path("/kaggle/working/lunar_lander_multieps_outputs.zip")
output_directories = ("results", "img", "logs", "policy")

with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:
    for directory_name in output_directories:
        output_directory = WORK_DIR / directory_name
        if output_directory.exists():
            for output_path in sorted(output_directory.rglob("*")):
                if output_path.is_file():
                    archive.write(
                        output_path, output_path.relative_to(WORK_DIR)
                    )
    for configuration_name in ("trajectory.json", "abstraction.json"):
        configuration_path = WORK_DIR / configuration_name
        if configuration_path.is_file():
            archive.write(configuration_path, configuration_path.name)

print(f"Output archive ready: {archive_path}")
print(f"Archive size: {archive_path.stat().st_size / (1024 ** 2):.2f} MB")
